# Parent-group penalty sensitivity analysis

This notebook extends `extended_monte_carlo_model_selection.ipynb` with a parent-group penalty sensitivity analysis. It compares component-wise Lasso (`glmnet`, `alpha = 1`) with parent-group Lasso (`grpreg`, `penalty = "grLasso"`) while preserving the original Monte Carlo data-generating process, preprocessing, sparsification schemes, controlled lambda grids, parent-level evaluation rules, scenario IDs, and random seeds.

The implementation is designed for reproducibility. At execution time it checks numerical reproduction against the existing **16,000-row baseline results CSV** in `outputs/extended_monte_carlo/` before producing new summaries.

**Start with `RUN_MODE <- "quick"`.** Run all cells from a fresh R kernel in the project root. Keep the original `outputs/extended_monte_carlo/` folder available. Quick mode uses canonical scenarios 1 and 11, two replications, all four interval schemes, and both path specifications. Quick-mode outputs are validation artifacts only. After all checks pass, change only `RUN_MODE` to `"full"`, restart R, and run all cells. Full mode uses all 16 canonical scenarios and 100 replications.

## Analysis objective

Expanding a parent feature into interval-specific components changes both the representation and the way regularization acts on that representation. This analysis tests how the reported sparse-versus-original results change when interval components belonging to the same parent feature are coupled through a group penalty.

The grouped analysis is a **sensitivity analysis**, not an algebraically exact reconstruction of the original parent-feature Lasso penalty and not a complete causal decomposition of representation and penalty effects.

`grpreg` standardizes columns and orthonormalizes groups internally. With centered group designs, its transformed group norm corresponds to $\|X_j\beta_j\|_2/\sqrt{n}$. The primary group multipliers are all one. Optional square-root group-size weights are included as an additional specification; when all groups have equal size, rescaling all weights can be absorbed into the normalized lambda path and therefore does not provide an independent robustness result.

Evaluation uses **matched target parent-feature sizes**, `s` and `2s`. Realized counts can differ because a path can overshoot or fail to reach a target. Parent counts, component counts, target attainment, and clearly labeled predictor-only degrees-of-freedom diagnostics are exported.

Kendall and Jaccard measures compare original and expanded representations **within each method**. They are distinct from the original study's agreement measures among Lasso, LAR, and glmnet within a representation. The simulated response remains linear in the original predictors, so the original representation is correctly specified by construction.

## Reproducibility requirements

- Run from the same project root as `extended_monte_carlo_model_selection.ipynb`.
- Preserve the archived baseline file at `outputs/extended_monte_carlo/simulation_replication_results_controlled_paths.csv`.
- Use the documented R/package environment when possible.
- Do not interpret full-run results unless all preflight, path, coverage, and baseline-reproduction checks pass.


In [1]:
install.packages(
  "grpreg",
  repos = "https://cloud.r-project.org",
  dependencies = TRUE
)

Installing package into 'C:/Users/it08d/AppData/Local/R/win-library/4.4'
(as 'lib' is unspecified)



package 'grpreg' successfully unpacked and MD5 sums checked

The downloaded binary packages are in
	C:\Users\it08d\AppData\Local\Temp\RtmpyS0Abu\downloaded_packages


In [2]:
# =============================================================================
# 1. SETUP AND RUN CONTROLS
# =============================================================================

options(stringsAsFactors = FALSE)

required_packages <- c("MASS", "glmnet", "grpreg")
missing_packages <- required_packages[
  !vapply(required_packages, requireNamespace, logical(1), quietly = TRUE)
]

if (length(missing_packages) > 0L) {
  stop(
    paste0(
      "Missing packages: ", paste(missing_packages, collapse = ", "),
      ". Install them first, e.g. install.packages(c(",
      paste(sprintf("'%s'", missing_packages), collapse = ", "), "))."
    )
  )
}

suppressPackageStartupMessages({
  library(MASS)
  library(glmnet)
  library(grpreg)
})

cat("R version:", R.version.string, "\n")
cat("MASS version:", as.character(packageVersion("MASS")), "\n")
cat("glmnet version:", as.character(packageVersion("glmnet")), "\n")
cat("grpreg version:", as.character(packageVersion("grpreg")), "\n")

# ---------------------------- Run controls ----------------------------------
RUN_MODE <- "full"                # Change to "full" only after a successful quick run.
RUN_EXPERIMENT <- TRUE
RUN_PATH_SENSITIVITY <- TRUE
RUN_GROUP_WEIGHT_SENSITIVITY <- TRUE
NOTEBOOK_VERSION <- "2026-09-17-v1"
ITERATION_BUDGET_MULTIPLIERS <- c(1L, 5L, 25L)
REFERENCE_FILE <- file.path("outputs", "extended_monte_carlo",
                            "simulation_replication_results_controlled_paths.csv")
REFERENCE_F1_TOLERANCE <- 1e-10
REFERENCE_RMSE_TOLERANCE <- 1e-8
SINGLETON_PREDICTION_TOLERANCE <- 5e-3 # max pathwise training RMS difference / sd(y)
stopifnot(RUN_MODE %in% c("quick", "full"))
# Original source used set.seed() under R 4.4.1 defaults. Reproduction checks
# verify this assumption; do not relax their tolerances to hide a mismatch.
RNGkind("Mersenne-Twister", "Inversion", "Rejection")

BASE_SEED <- 20260711L
SIGNAL_TO_NOISE_RATIO <- 3
TEST_SAMPLE_SIZE <- 500L
COEFFICIENT_TOLERANCE <- 1e-8

PRIMARY_PATH_SPECIFICATION <- "primary_controlled"
PATH_SPECIFICATIONS <- data.frame(
  path_specification = c(PRIMARY_PATH_SPECIFICATION, "sensitivity_dense_wide"),
  nlambda = c(200L, 400L),
  lambda_min_ratio = c(1e-2, 1e-3),
  thresh = c(1e-8, 1e-10),
  maxit = c(100000L, 100000L),
  disable_early_stopping = c(TRUE, TRUE),
  type_gaussian = c("naive", "naive"),
  grpreg_eps = c(1e-6, 1e-8),
  grpreg_max_iter = c(100000L, 200000L),
  stringsAsFactors = FALSE
)
if (!RUN_PATH_SENSITIVITY) {
  PATH_SPECIFICATIONS <- PATH_SPECIFICATIONS[
    PATH_SPECIFICATIONS$path_specification == PRIMARY_PATH_SPECIFICATION, , drop = FALSE
  ]
}

NUMBER_OF_REPLICATIONS <- if (RUN_MODE == "full") 100L else 2L
# Always build this complete design before selecting quick cases.
NP_SETTINGS <- data.frame(n = c(100L, 100L, 100L, 500L),
                          p = c(20L, 100L, 200L, 100L))
CORRELATION_SETTINGS <- c("independent", "ar1_rho_0.5", "ar1_rho_0.9", "block")
SPARSIFICATION_SCHEMES <- c("quantile_2", "quantile_4", "quantile_8", "equal_width_4")

GROUP_WEIGHT_SPECIFICATIONS <- c("equal_parent")
if (RUN_GROUP_WEIGHT_SENSITIVITY) {
  GROUP_WEIGHT_SPECIFICATIONS <- c(
    GROUP_WEIGHT_SPECIFICATIONS, "sqrt_group_size"
  )
}

OUTPUT_ROOT <- file.path("outputs", "parent_group_penalty_sensitivity")
# OUTPUT_DIRECTORY is initialized after all analysis functions are defined.
output_path <- function(filename) file.path(OUTPUT_DIRECTORY, filename)

make_lambda_fraction_grid <- function(nlambda, lambda_min_ratio) {
  exp(seq(from = 0, to = log(lambda_min_ratio), length.out = nlambda))
}

cat("Run mode:", RUN_MODE, "; replications:", NUMBER_OF_REPLICATIONS, "\n")
cat("Original recorded environment: R 4.4.1; MASS 7.3-60.2; glmnet 4.1-10.\n")
cat("Use that environment when possible; install grpreg there without updating the baseline packages.\n")
print(PATH_SPECIFICATIONS)


Warning message:
"package 'glmnet' was built under R version 4.4.3"
Warning message:
"package 'grpreg' was built under R version 4.4.3"


R version: R version 4.4.1 (2024-06-14 ucrt) 
MASS version: 7.3.60.2 
glmnet version: 4.1.10 
grpreg version: 3.6.0 
Run mode: full ; replications: 100 
Original recorded environment: R 4.4.1; MASS 7.3-60.2; glmnet 4.1-10.
Use that environment when possible; install grpreg there without updating the baseline packages.
      path_specification nlambda lambda_min_ratio thresh  maxit
1     primary_controlled     200            0.010  1e-08 100000
2 sensitivity_dense_wide     400            0.001  1e-10 100000
  disable_early_stopping type_gaussian grpreg_eps grpreg_max_iter
1                   TRUE         naive      1e-06          100000
2                   TRUE         naive      1e-08          200000


## 2. Original simulation and transformation

The following simulation, standardization, and interval-expansion functions are unchanged from the original Monte Carlo notebook. Canonical scenario IDs are assigned before subsetting quick mode, preserving the seed formula `BASE_SEED + scenario_id * 100000 + replication`. Test sample size remains 500 even in validation because changing it would change subsequent random draws in the original generator.


In [3]:
# =============================================================================
# 2. SIMULATION DESIGN
# =============================================================================

design_list <- list()
design_counter <- 0L
for (np_index in seq_len(nrow(NP_SETTINGS))) {
  for (correlation_name in CORRELATION_SETTINGS) {
    design_counter <- design_counter + 1L
    current_n <- NP_SETTINGS$n[np_index]
    current_p <- NP_SETTINGS$p[np_index]
    design_list[[design_counter]] <- data.frame(
      n = current_n,
      p = current_p,
      dimension = paste0("n=", current_n, ", p=", current_p),
      correlation = correlation_name,
      active_s = max(5L, floor(0.10 * current_p)),
      stringsAsFactors = FALSE
    )
  }
}
simulation_design <- do.call(rbind, design_list)
simulation_design$scenario_id <- seq_len(nrow(simulation_design))
simulation_design <- simulation_design[, c(
  "scenario_id", "n", "p", "dimension", "correlation", "active_s"
)]
rownames(simulation_design) <- NULL
canonical_design <- simulation_design
if (RUN_MODE == "quick") {
  simulation_design <- canonical_design[canonical_design$scenario_id %in% c(1L, 11L), , drop = FALSE]
}
print(simulation_design)

# ---- Copied from extended_monte_carlo_model_selection.ipynb ----------------
make_covariance_matrix <- function(
    p,
    correlation_structure) {

  p <- as.integer(p)

  if (correlation_structure == "independent") {

    covariance_matrix <- diag(p)

  } else if (correlation_structure == "ar1_rho_0.5") {

    feature_index <- seq_len(p)

    covariance_matrix <- 0.5^abs(
      outer(
        feature_index,
        feature_index,
        "-"
      )
    )

  } else if (correlation_structure == "ar1_rho_0.9") {

    feature_index <- seq_len(p)

    covariance_matrix <- 0.9^abs(
      outer(
        feature_index,
        feature_index,
        "-"
      )
    )

  } else if (correlation_structure == "block") {

    block_size <- 10L
    within_block_correlation <- 0.8
    between_block_correlation <- 0.1

    covariance_matrix <- matrix(
      between_block_correlation,
      nrow = p,
      ncol = p
    )

    block_starts <- seq.int(
      from = 1L,
      to = p,
      by = block_size
    )

    for (block_start in block_starts) {

      block_end <- min(
        block_start + block_size - 1L,
        p
      )

      block_indices <- block_start:block_end

      covariance_matrix[
        block_indices,
        block_indices
      ] <- within_block_correlation
    }

    diag(covariance_matrix) <- 1

  } else {

    stop(
      "Unknown correlation structure: ",
      correlation_structure
    )
  }

  minimum_eigenvalue <- min(
    eigen(
      covariance_matrix,
      symmetric = TRUE,
      only.values = TRUE
    )$values
  )

  if (minimum_eigenvalue <= 1e-8) {

    covariance_matrix <- covariance_matrix +
      diag(
        abs(minimum_eigenvalue) + 1e-6,
        p
      )
  }

  covariance_matrix
}


generate_regression_data <- function(
    n,
    p,
    correlation,
    signal_to_noise_ratio = 3,
    test_sample_size = 500L,
    seed = 1L) {

  set.seed(seed)

  covariance_matrix <- make_covariance_matrix(
    p = p,
    correlation_structure = correlation
  )

  active_s <- max(
    5L,
    floor(0.10 * p)
  )

  true_support <- sort(
    sample.int(
      p,
      size = active_s,
      replace = FALSE
    )
  )

  beta <- numeric(p)

  beta[true_support] <- sample(
    c(-1, 1),
    size = active_s,
    replace = TRUE
  ) * runif(
    active_s,
    min = 0.5,
    max = 1.5
  )

  X_train <- MASS::mvrnorm(
    n = n,
    mu = rep(0, p),
    Sigma = covariance_matrix
  )

  X_test <- MASS::mvrnorm(
    n = test_sample_size,
    mu = rep(0, p),
    Sigma = covariance_matrix
  )

  training_signal <- drop(
    X_train %*% beta
  )

  test_signal <- drop(
    X_test %*% beta
  )

  signal_variance <- var(
    training_signal
  )

  noise_standard_deviation <- sqrt(
    signal_variance / signal_to_noise_ratio
  )

  y_train <- training_signal + rnorm(
    n,
    mean = 0,
    sd = noise_standard_deviation
  )

  y_test <- test_signal + rnorm(
    test_sample_size,
    mean = 0,
    sd = noise_standard_deviation
  )

  list(
    X_train = X_train,
    X_test = X_test,
    y_train = y_train,
    y_test = y_test,
    beta = beta,
    true_support = true_support,
    active_s = active_s,
    noise_sd = noise_standard_deviation,
    covariance_matrix = covariance_matrix
  )
}



fit_standardizer <- function(
    X,
    tolerance = 1e-12) {

  X <- as.matrix(X)

  feature_means <- colMeans(X)

  feature_standard_deviations <- apply(
    X,
    2L,
    sd
  )

  valid_features <- is.finite(
    feature_standard_deviations
  ) &
    feature_standard_deviations > tolerance

  if (!all(valid_features)) {
    stop(
      "At least one generated original feature has zero variance."
    )
  }

  list(
    center = feature_means,
    scale = feature_standard_deviations,
    keep = valid_features
  )
}


apply_standardizer <- function(
    X,
    standardizer) {

  X <- as.matrix(X)

  X <- X[
    ,
    standardizer$keep,
    drop = FALSE
  ]

  X <- sweep(
    X,
    2L,
    standardizer$center,
    "-"
  )

  X <- sweep(
    X,
    2L,
    standardizer$scale,
    "/"
  )

  X
}


parse_sparsification_scheme <- function(scheme) {

  if (grepl("^quantile_", scheme)) {

    list(
      type = "quantile",
      bins = as.integer(
        sub(
          "^quantile_",
          "",
          scheme
        )
      )
    )

  } else if (grepl("^equal_width_", scheme)) {

    list(
      type = "equal_width",
      bins = as.integer(
        sub(
          "^equal_width_",
          "",
          scheme
        )
      )
    )

  } else {

    stop(
      "Unknown sparsification scheme: ",
      scheme
    )
  }
}


make_training_breaks <- function(
    x,
    scheme_type,
    number_of_bins) {

  if (scheme_type == "quantile") {

    raw_breaks <- as.numeric(
      quantile(
        x,
        probs = seq(
          0,
          1,
          length.out = number_of_bins + 1L
        ),
        type = 8,
        names = FALSE,
        na.rm = TRUE
      )
    )

  } else if (scheme_type == "equal_width") {

    raw_breaks <- seq(
      min(x),
      max(x),
      length.out = number_of_bins + 1L
    )

  } else {

    stop(
      "Unknown sparsification type: ",
      scheme_type
    )
  }

  raw_breaks <- unique(
    raw_breaks
  )

  if (length(raw_breaks) <= 2L) {
    return(
      c(-Inf, Inf)
    )
  }

  internal_breaks <- raw_breaks[
    2L:(length(raw_breaks) - 1L)
  ]

  c(
    -Inf,
    internal_breaks,
    Inf
  )
}


fit_sparsifier <- function(
    X_train_standardized,
    scheme,
    tolerance = 1e-12) {

  X_train_standardized <- as.matrix(
    X_train_standardized
  )

  specification <- parse_sparsification_scheme(
    scheme
  )

  raw_components <- list()
  component_information <- list()
  component_counter <- 0L

  for (
    parent_index in
    seq_len(ncol(X_train_standardized))
  ) {

    parent_values <- X_train_standardized[
      ,
      parent_index
    ]

    breaks <- make_training_breaks(
      x = parent_values,
      scheme_type = specification$type,
      number_of_bins = specification$bins
    )

    interval_membership <- cut(
      parent_values,
      breaks = breaks,
      include.lowest = TRUE,
      right = TRUE,
      labels = FALSE
    )

    observed_intervals <- sort(
      unique(
        interval_membership[
          !is.na(interval_membership)
        ]
      )
    )

    for (interval_index in observed_intervals) {

      component_counter <- component_counter + 1L

      component_values <- ifelse(
        interval_membership == interval_index,
        parent_values,
        0
      )

      raw_components[[component_counter]] <- component_values

      component_information[[component_counter]] <- data.frame(
        component = component_counter,
        parent = parent_index,
        interval = interval_index,
        lower = breaks[interval_index],
        upper = breaks[interval_index + 1L],
        stringsAsFactors = FALSE
      )
    }
  }

  X_raw <- do.call(
    cbind,
    raw_components
  )

  component_table <- do.call(
    rbind,
    component_information
  )

  component_centers <- colMeans(
    X_raw
  )

  component_scales <- apply(
    X_raw,
    2L,
    sd
  )

  keep_components <- is.finite(
    component_scales
  ) &
    component_scales > tolerance

  if (!any(keep_components)) {
    stop(
      "Sparsification generated no nonconstant components."
    )
  }

  X_retained <- X_raw[
    ,
    keep_components,
    drop = FALSE
  ]

  X_scaled <- sweep(
    X_retained,
    2L,
    component_centers[keep_components],
    "-"
  )

  X_scaled <- sweep(
    X_scaled,
    2L,
    component_scales[keep_components],
    "/"
  )

  retained_information <- component_table[
    keep_components,
    ,
    drop = FALSE
  ]

  rownames(retained_information) <- NULL

  list(
    X_train = X_scaled,
    scheme = scheme,
    type = specification$type,
    requested_bins = specification$bins,
    component_table = retained_information,
    component_center = component_centers[keep_components],
    component_scale = component_scales[keep_components],
    parent_map = as.integer(retained_information$parent),
    raw_density = mean(
      abs(X_retained) > tolerance
    )
  )
}


apply_sparsifier <- function(
    X_standardized,
    fitted_sparsifier) {

  X_standardized <- as.matrix(
    X_standardized
  )

  component_table <- fitted_sparsifier$component_table

  X_raw <- matrix(
    0,
    nrow = nrow(X_standardized),
    ncol = nrow(component_table)
  )

  for (
    component_index in
    seq_len(nrow(component_table))
  ) {

    parent_index <- component_table$parent[
      component_index
    ]

    lower_bound <- component_table$lower[
      component_index
    ]

    upper_bound <- component_table$upper[
      component_index
    ]

    parent_values <- X_standardized[
      ,
      parent_index
    ]

    belongs_to_interval <- (
      parent_values > lower_bound &
        parent_values <= upper_bound
    )

    X_raw[
      ,
      component_index
    ] <- ifelse(
      belongs_to_interval,
      parent_values,
      0
    )
  }

  X_scaled <- sweep(
    X_raw,
    2L,
    fitted_sparsifier$component_center,
    "-"
  )

  X_scaled <- sweep(
    X_scaled,
    2L,
    fitted_sparsifier$component_scale,
    "/"
  )

  X_scaled
}


   scenario_id   n   p    dimension correlation active_s
1            1 100  20  n=100, p=20 independent        5
2            2 100  20  n=100, p=20 ar1_rho_0.5        5
3            3 100  20  n=100, p=20 ar1_rho_0.9        5
4            4 100  20  n=100, p=20       block        5
5            5 100 100 n=100, p=100 independent       10
6            6 100 100 n=100, p=100 ar1_rho_0.5       10
7            7 100 100 n=100, p=100 ar1_rho_0.9       10
8            8 100 100 n=100, p=100       block       10
9            9 100 200 n=100, p=200 independent       20
10          10 100 200 n=100, p=200 ar1_rho_0.5       20
11          11 100 200 n=100, p=200 ar1_rho_0.9       20
12          12 100 200 n=100, p=200       block       20
13          13 500 100 n=500, p=100 independent       10
14          14 500 100 n=500, p=100 ar1_rho_0.5       10
15          15 500 100 n=500, p=100 ar1_rho_0.9       10
16          16 500 100 n=500, p=100       block       10


## 3. Controlled component-wise Lasso path

This is the same explicit `glmnet` path construction used in the controlled Monte Carlo analysis: standardized inputs, centered training response, `alpha = 1`, no internal standardization or intercept, explicit representation-specific `lambda_max`, and common normalized lambda fractions.


In [4]:
# =============================================================================
# 3. CONTROLLED GLMNET PATH (same implementation as existing analysis)
# =============================================================================
extract_glmnet_path <- function(
    fitted_model,
    number_of_features) {

  coefficient_path <- t(
    as.matrix(fitted_model$beta)
  )

  if (ncol(coefficient_path) != number_of_features) {
    stop(
      "Unexpected glmnet coefficient-path dimension."
    )
  }

  rbind(
    rep(0, number_of_features),
    coefficient_path
  )
}


make_glmnet_lambda_sequence <- function(
    X,
    centered_y,
    nlambda,
    lambda_min_ratio) {

  X <- as.matrix(X)
  centered_y <- as.numeric(centered_y)

  if (nlambda < 2L) {
    stop("nlambda must be at least 2.")
  }

  if (
    !is.finite(lambda_min_ratio) ||
      lambda_min_ratio <= 0 ||
      lambda_min_ratio >= 1
  ) {
    stop(
      "lambda_min_ratio must be strictly between 0 and 1."
    )
  }

  lambda_max <- max(
    abs(
      drop(
        crossprod(
          X,
          centered_y
        )
      )
    )
  ) / nrow(X)

  if (
    !is.finite(lambda_max) ||
      lambda_max <= .Machine$double.eps
  ) {
    stop(
      "Could not construct a positive finite lambda_max."
    )
  }

  lambda_fractions <- make_lambda_fraction_grid(
    nlambda = nlambda,
    lambda_min_ratio = lambda_min_ratio
  )

  lambda_sequence <- lambda_max *
    lambda_fractions

  list(
    lambda = lambda_sequence,
    lambda_max = lambda_max,
    lambda_min = min(lambda_sequence),
    lambda_min_ratio = min(lambda_sequence) /
      max(lambda_sequence)
  )
}


fit_glmnet_controlled <- function(
    X,
    centered_y,
    nlambda,
    lambda_min_ratio,
    thresh,
    maxit,
    disable_early_stopping,
    type_gaussian) {

  X <- as.matrix(X)

  lambda_information <-
    make_glmnet_lambda_sequence(
      X = X,
      centered_y = centered_y,
      nlambda = nlambda,
      lambda_min_ratio = lambda_min_ratio
    )

  current_glmnet_control <-
    glmnet::glmnet.control()

  per_fit_control <- list(
    thresh = thresh,
    maxit = as.integer(maxit),
    dfmax = ncol(X),
    pmax = ncol(X)
  )

  if (isTRUE(disable_early_stopping)) {

    # Per-fit path controls avoid changing global/session state when the
    # installed glmnet version supports the `control` argument.
    # fdev = 0 and devmax = 1 disable the usual fractional-deviance
    # and near-saturation stopping rules. mnlam = nlambda ensures that
    # early-stopping checks cannot truncate the requested explicit grid.
    per_fit_control$fdev <- 0
    per_fit_control$devmax <- 1
    per_fit_control$mnlam <-
      as.integer(nlambda)
  }

  glmnet_formal_arguments <- names(
    formals(
      glmnet::glmnet
    )
  )

  if (
    "control" %in%
      glmnet_formal_arguments
  ) {

    glmnet_control_interface <-
      "per_fit_control"

    fitted_model <- glmnet::glmnet(
      X,
      centered_y,
      alpha = 1,
      family = "gaussian",
      standardize = FALSE,
      intercept = FALSE,
      lambda = lambda_information$lambda,
      type.gaussian = type_gaussian,
      control = per_fit_control
    )

  } else {

    # Compatibility branch for older glmnet releases that do not expose
    # per-fit control=. The same numerical/path controls are applied using
    # the legacy arguments and session controls, and the previous controls
    # are restored on exit.
    glmnet_control_interface <-
      "legacy_arguments"

    previous_control <-
      glmnet::glmnet.control()

    control_formals <- names(
      formals(
        glmnet::glmnet.control
      )
    )

    restore_names <- intersect(
      names(previous_control),
      setdiff(
        control_formals,
        "factory"
      )
    )

    on.exit(
      do.call(
        glmnet::glmnet.control,
        previous_control[
          restore_names
        ]
      ),
      add = TRUE
    )

    if (
      isTRUE(
        disable_early_stopping
      )
    ) {
      glmnet::glmnet.control(
        fdev = 0,
        devmax = 1,
        mnlam = as.integer(nlambda)
      )
    }

    fitted_model <- glmnet::glmnet(
      X,
      centered_y,
      alpha = 1,
      family = "gaussian",
      standardize = FALSE,
      intercept = FALSE,
      lambda = lambda_information$lambda,
      thresh = thresh,
      maxit = as.integer(maxit),
      dfmax = ncol(X),
      pmax = ncol(X),
      type.gaussian = type_gaussian
    )
  }

  beta_matrix <-
    as.matrix(
      fitted_model$beta
    )

  diagnostics <- list(
    requested_lambda_points =
      as.integer(nlambda),
    returned_lambda_points =
      length(fitted_model$lambda),
    requested_lambda_min_ratio =
      lambda_min_ratio,
    requested_lambda_max =
      lambda_information$lambda_max,
    requested_lambda_min =
      lambda_information$lambda_min,
    returned_lambda_max =
      max(fitted_model$lambda),
    returned_lambda_min =
      min(fitted_model$lambda),
    returned_lambda_min_ratio =
      min(fitted_model$lambda) /
        max(fitted_model$lambda),
    thresh = thresh,
    maxit = as.integer(maxit),
    disable_early_stopping =
      isTRUE(disable_early_stopping),
    fdev = if (
      isTRUE(disable_early_stopping)
    ) {
      0
    } else {
      current_glmnet_control$fdev
    },
    devmax = if (
      isTRUE(disable_early_stopping)
    ) {
      1
    } else {
      current_glmnet_control$devmax
    },
    mnlam = if (
      isTRUE(disable_early_stopping)
    ) {
      as.integer(nlambda)
    } else {
      current_glmnet_control$mnlam
    },
    dfmax = ncol(X),
    pmax = ncol(X),
    type_gaussian = type_gaussian,
    control_interface =
      glmnet_control_interface,
    jerr = fitted_model$jerr,
    first_lambda_active_features =
      sum(
        abs(
          beta_matrix[
            ,
            1L
          ]
        ) > 1e-10
      ),
    last_lambda_active_features =
      sum(
        abs(
          beta_matrix[
            ,
            ncol(beta_matrix)
          ]
        ) > 1e-10
      ),
    complete_requested_grid =
      length(fitted_model$lambda) ==
        as.integer(nlambda)
  )

  list(
    model = fitted_model,
    diagnostics = diagnostics
  )
}


## 4. Controlled paths, convergence, and complexity diagnostics

The component-Lasso function in Section 3 remains unchanged. A wrapper captures warnings and checks optimizer status, finite coefficients, every requested lambda value, and path length. Failed fits are retried with iteration budgets multiplied by 5 and then 25, keeping data, lambda range, and tolerances fixed. Every attempt is recorded. A path that still fails stops the run; partial paths are not silently summarized.

For group Lasso, the automatic log grid is checked against the requested normalized grid. Its `max.iter` is a budget across the whole path. Returned iteration counts and warnings are checked and exported. Requested and accepted budgets are retained. The singleton-group comparison also checks lambda-scale calibration and fitted predictions at aligned normalized path points.

The grouped penalty uses `grpreg`'s orthonormalized group scale, not simply the Euclidean norm of coefficients in the supplied component matrix. See [the authors' description, Section 2.1](https://pbreheny.org/web/assets/group-computing.pdf) and [the package documentation](https://pbreheny.github.io/grpreg/reference/grpreg.html).

The `df_diagnostic` fields exclude the intercept. For glmnet they contain an active-coefficient-count **proxy**; for grpreg they contain its approximate degrees of freedom minus one for the intercept. A `df_definition` column accompanies these different diagnostics. They do not establish equal effective complexity. The intercept convention follows the [Gaussian package implementation](https://rdrr.io/cran/grpreg/src/R/grpreg.R).


In [5]:
# 4. Validated solver wrappers. Baseline glmnet implementation above is unchanged.

capture_fit <- function(expression) {
  messages <- character()
  error_text <- NULL
  value <- tryCatch(withCallingHandlers(
    force(expression),
    warning = function(w) {
      messages <<- c(messages, conditionMessage(w))
      invokeRestart("muffleWarning")
    }
  ), error = function(e) {
    error_text <<- conditionMessage(e)
    NULL
  })
  list(value = value, warnings = unique(messages), error = error_text)
}

grid_matches <- function(lambda, nlambda, lambda_min_ratio, expected = NULL) {
  if (length(lambda) != nlambda || any(!is.finite(lambda)) ||
      any(lambda <= 0) || any(diff(lambda) >= 0)) return(FALSE)
  if (is.null(expected)) expected <- lambda[1L] * make_lambda_fraction_grid(nlambda, lambda_min_ratio)
  length(expected) == length(lambda) &&
    max(abs(lambda / expected - 1)) < 1e-8
}

stop_failed_fit <- function(method, attempts) {
  detail <- paste(attempts, collapse = " || ")
  stop(structure(list(message = paste(method, "path rejected:", detail),
                      call = NULL, attempts = attempts),
                 class = c("penalty_control_fit_error", "error", "condition")))
}

common_diagnostics <- function(lambda, coefficient_matrix, nlambda, ratio,
                               requested_maxit, accepted_maxit, attempts,
                               warnings, jerr = NA_integer_, npasses = NA_real_,
                               total_iterations = NA_real_, eps = NA_real_,
                               thresh = NA_real_, control_interface = "grpreg") {
  list(
    requested_lambda_points = as.integer(nlambda),
    returned_lambda_points = length(lambda),
    requested_lambda_min_ratio = ratio,
    returned_lambda_min_ratio = tail(lambda, 1L) / lambda[1L],
    returned_lambda_max = lambda[1L],
    returned_lambda_min = tail(lambda, 1L),
    complete_requested_grid = TRUE,
    accepted_path = TRUE,
    requested_iteration_budget = as.integer(requested_maxit),
    accepted_iteration_budget = as.integer(accepted_maxit),
    attempt_count = length(attempts),
    attempt_log = paste(attempts, collapse = " || "),
    warnings = paste(warnings, collapse = " | "),
    jerr = jerr,
    glmnet_npasses = npasses,
    grpreg_total_iterations = total_iterations,
    grpreg_eps = eps,
    glmnet_thresh = thresh,
    control_interface = control_interface,
    first_lambda_active_components = sum(abs(coefficient_matrix[, 1L]) > COEFFICIENT_TOLERANCE),
    last_lambda_active_components = sum(abs(coefficient_matrix[, ncol(coefficient_matrix)]) > COEFFICIENT_TOLERANCE)
  )
}

make_group_multiplier <- function(parent_map, number_of_parents, specification) {
  group_sizes <- tabulate(parent_map, nbins = number_of_parents)
  if (any(group_sizes == 0L)) stop("A parent feature has no retained components.")
  if (specification == "equal_parent") return(rep(1, number_of_parents))
  if (specification == "sqrt_group_size") return(sqrt(group_sizes))
  stop("Unknown group-weight specification: ", specification)
}

fit_group_lasso_controlled <- function(
    X, centered_y, parent_map, number_of_parents, nlambda, lambda_min_ratio,
    eps, maxit, group_weight_specification) {
  X <- as.matrix(X)
  parent_map <- as.integer(parent_map)
  stopifnot(ncol(X) == length(parent_map), all(is.finite(X)),
            all(is.finite(centered_y)), all(parent_map >= 1L),
            all(parent_map <= number_of_parents))
  group_multiplier <- make_group_multiplier(parent_map, number_of_parents,
                                             group_weight_specification)
  attempts <- character()
  all_warnings <- character()
  for (multiplier in ITERATION_BUDGET_MULTIPLIERS) {
    budget <- as.integer(maxit * multiplier)
    captured <- capture_fit(grpreg::grpreg(
      X = X, y = as.numeric(centered_y), group = parent_map,
      penalty = "grLasso", family = "gaussian",
      nlambda = as.integer(nlambda), lambda.min = lambda_min_ratio,
      log.lambda = TRUE, alpha = 1, eps = eps, max.iter = budget,
      dfmax = ncol(X), gmax = number_of_parents,
      group.multiplier = group_multiplier, warn = TRUE, returnX = FALSE
    ))
    all_warnings <- unique(c(all_warnings, captured$warnings))
    if (is.null(captured$value)) {
      attempts <- c(attempts, paste0("budget=", budget, "; error=", captured$error))
      next
    }
    model <- captured$value
    beta <- as.matrix(model$beta)
    shape_ok <- nrow(beta) == ncol(X) + 1L && ncol(beta) == length(model$lambda)
    grid_ok <- grid_matches(model$lambda, nlambda, lambda_min_ratio)
    iter_ok <- length(model$iter) == nlambda && all(is.finite(model$iter)) &&
      all(model$iter >= 0) && sum(model$iter) < budget
    df_ok <- length(model$df) == nlambda && all(is.finite(model$df)) &&
      all(model$df >= 1 - 1e-8)
    ok <- shape_ok && grid_ok && iter_ok && df_ok &&
      all(is.finite(beta)) && length(captured$warnings) == 0L
    attempts <- c(attempts, paste0(
      "budget=", budget, "; points=", length(model$lambda), "/", nlambda,
      "; grid=", grid_ok, "; iter_sum=", sum(model$iter),
      "; df_ok=", df_ok, "; warnings=", paste(captured$warnings, collapse = " | "),
      "; accepted=", ok))
    if (!ok) next
    coefficient_matrix <- beta[-1L, , drop = FALSE]
    return(list(
      model = model,
      coefficient_path = rbind(rep(0, ncol(X)), t(coefficient_matrix)),
      intercept_path = c(0, as.numeric(beta[1L, ])),
      df_diagnostic = c(0, pmax(0, as.numeric(model$df) - 1)),
      df_definition = "grpreg_approximate_df_excluding_intercept",
      diagnostics = common_diagnostics(
        model$lambda, coefficient_matrix, nlambda, lambda_min_ratio,
        maxit, budget, attempts, all_warnings,
        total_iterations = sum(model$iter), eps = eps
      )
    ))
  }
  stop_failed_fit(paste("grpreg", group_weight_specification), attempts)
}

extract_component_lasso_control <- function(
    X, centered_y, nlambda, lambda_min_ratio, thresh, maxit,
    disable_early_stopping, type_gaussian) {
  X <- as.matrix(X)
  stopifnot(all(is.finite(X)), all(is.finite(centered_y)))
  expected <- make_glmnet_lambda_sequence(X, centered_y, nlambda, lambda_min_ratio)$lambda
  attempts <- character()
  all_warnings <- character()
  for (multiplier in ITERATION_BUDGET_MULTIPLIERS) {
    budget <- as.integer(maxit * multiplier)
    captured <- capture_fit(fit_glmnet_controlled(
      X = X, centered_y = centered_y, nlambda = nlambda,
      lambda_min_ratio = lambda_min_ratio, thresh = thresh, maxit = budget,
      disable_early_stopping = disable_early_stopping, type_gaussian = type_gaussian
    ))
    all_warnings <- unique(c(all_warnings, captured$warnings))
    if (is.null(captured$value)) {
      attempts <- c(attempts, paste0("budget=", budget, "; error=", captured$error))
      next
    }
    fit <- captured$value
    model <- fit$model
    beta <- as.matrix(model$beta)
    grid_ok <- grid_matches(model$lambda, nlambda, lambda_min_ratio, expected)
    status_ok <- length(model$jerr) == 1L && is.finite(model$jerr) && model$jerr == 0
    shape_ok <- nrow(beta) == ncol(X) && ncol(beta) == length(model$lambda)
    ok <- grid_ok && status_ok && shape_ok && all(is.finite(beta)) &&
      all(is.finite(model$a0)) && length(captured$warnings) == 0L
    attempts <- c(attempts, paste0(
      "budget=", budget, "; points=", length(model$lambda), "/", nlambda,
      "; grid=", grid_ok, "; jerr=", paste(model$jerr, collapse = ","),
      "; warnings=", paste(captured$warnings, collapse = " | "), "; accepted=", ok))
    if (!ok) next
    return(list(
      model = model,
      coefficient_path = extract_glmnet_path(model, ncol(X)),
      intercept_path = rep(0, length(model$lambda) + 1L),
      df_diagnostic = c(0, unname(colSums(abs(beta) > COEFFICIENT_TOLERANCE))),
      df_definition = "active_coefficient_count_proxy_excluding_intercept",
      diagnostics = common_diagnostics(
        model$lambda, beta, nlambda, lambda_min_ratio, maxit, budget,
        attempts, all_warnings, jerr = model$jerr, npasses = model$npasses,
        thresh = thresh, control_interface = fit$diagnostics$control_interface
      )
    ))
  }
  stop_failed_fit("glmnet", attempts)
}

singleton_calibration <- function(component_fit, group_fit, X, y) {
  # Each original column has the same training SD=1. grpreg's RMS scale is
  # sqrt((n-1)/n), so lambda_group * RMS equals lambda_glmnet for this case.
  rms <- sqrt(colMeans(X^2))
  stopifnot(max(abs(rms / rms[1L] - 1)) < 1e-10)
  lambda_error <- max(abs(group_fit$model$lambda * rms[1L] /
                           component_fit$model$lambda - 1))
  coefficient_delta <- group_fit$coefficient_path - component_fit$coefficient_path
  prediction_delta <- sweep(X %*% t(coefficient_delta), 2L,
                            group_fit$intercept_path - component_fit$intercept_path, "+")
  prediction_error <- max(sqrt(colMeans(prediction_delta^2))) / sd(y)
  if (!is.finite(lambda_error) || lambda_error > 1e-8 ||
      !is.finite(prediction_error) || prediction_error > SINGLETON_PREDICTION_TOLERANCE) {
    stop("Singleton calibration failed: lambda relative error=", lambda_error,
         "; max prediction RMS/sd(y)=", prediction_error,
         ". Inspect solver conventions/convergence before interpreting grouped results.")
  }
  list(lambda_scale_relative_error = lambda_error,
       max_training_prediction_rms_over_sd_y = prediction_error)
}


## 5. Parent-level evaluation

The first path point reaching each target is selected; if it never reaches that target, the nearest available count is used, as in the original analysis. Both target attainment and exact equality are reported. F1 assesses parent support; RMSE also reflects fitted coefficients and shrinkage. Neither comparison holds effective complexity constant.

Entry-order Kendall tau and active-set Jaccard overlap below measure **representation preservation within a method**, not agreement among the original study's three procedures. Undefined Kendall correlations are retained as missing, and summary sample counts expose this. No undefined value is treated as perfect agreement.


In [6]:
# =============================================================================
# 5. COMMON METRICS (parent-level logic copied from existing analysis)
# =============================================================================
mean_or_na <- function(values) {

  values <- values[
    is.finite(values)
  ]

  if (length(values) == 0L) {
    NA_real_
  } else {
    mean(values)
  }
}


parent_entry_ranks <- function(
    coefficient_path,
    parent_map,
    number_of_parents,
    tolerance = 1e-8) {

  entry_step <- rep(
    Inf,
    number_of_parents
  )

  for (
    component_index in
    seq_len(ncol(coefficient_path))
  ) {

    active_steps <- which(
      abs(
        coefficient_path[
          ,
          component_index
        ]
      ) > tolerance
    )

    if (length(active_steps) > 0L) {

      first_active_step <- active_steps[1L]

      parent_index <- parent_map[
        component_index
      ]

      entry_step[parent_index] <- min(
        entry_step[parent_index],
        first_active_step
      )
    }
  }

  entry_step[
    !is.finite(entry_step)
  ] <- nrow(coefficient_path) + 1L

  rank(
    entry_step,
    ties.method = "average"
  )
}


path_point_at_size <- function(
    coefficient_path,
    parent_map,
    target_size,
    tolerance = 1e-8) {

  number_of_steps <- nrow(
    coefficient_path
  )

  model_sizes <- integer(
    number_of_steps
  )

  active_parent_sets <- vector(
    "list",
    number_of_steps
  )

  for (step_index in seq_len(number_of_steps)) {

    active_components <- which(
      abs(
        coefficient_path[
          step_index,
        ]
      ) > tolerance
    )

    active_parents <- sort(
      unique(
        parent_map[
          active_components
        ]
      )
    )

    active_parent_sets[[step_index]] <- active_parents

    model_sizes[step_index] <- length(
      active_parents
    )
  }

  eligible_steps <- which(
    model_sizes >= target_size
  )

  if (length(eligible_steps) > 0L) {

    selected_step <- eligible_steps[1L]

  } else {

    selected_step <- which.min(
      abs(
        model_sizes - target_size
      )
    )
  }

  list(
    step = selected_step,
    active_parents = active_parent_sets[[selected_step]],
    coefficients = coefficient_path[selected_step, ],
    selected_size = model_sizes[selected_step]
  )
}


jaccard_similarity <- function(
    first_set,
    second_set) {

  combined_set <- union(
    first_set,
    second_set
  )

  if (length(combined_set) == 0L) {
    return(1)
  }

  length(
    intersect(
      first_set,
      second_set
    )
  ) / length(combined_set)
}


calculate_support_metrics <- function(
    selected_support,
    true_support) {

  selected_support <- unique(
    selected_support
  )

  true_support <- unique(
    true_support
  )

  true_positives <- length(
    intersect(
      selected_support,
      true_support
    )
  )

  precision <- if (length(selected_support) == 0L) {
    0
  } else {
    true_positives / length(selected_support)
  }

  recall <- if (length(true_support) == 0L) {
    0
  } else {
    true_positives / length(true_support)
  }

  f1 <- if ((precision + recall) == 0) {
    0
  } else {
    2 * precision * recall /
      (precision + recall)
  }

  c(
    precision = precision,
    recall = recall,
    f1 = f1
  )
}


path_point_details <- function(
    coefficient_path,
    parent_map,
    target_size,
    df_diagnostic = NULL,
    tolerance = 1e-8) {

  selected <- path_point_at_size(
    coefficient_path = coefficient_path,
    parent_map = parent_map,
    target_size = target_size,
    tolerance = tolerance
  )
  component_count <- sum(abs(selected$coefficients) > tolerance)
  df_value <- if (is.null(df_diagnostic) || selected$step > length(df_diagnostic)) {
    NA_real_
  } else {
    unname(df_diagnostic[selected$step])
  }
  c(
    step = selected$step,
    parent_count = selected$selected_size,
    component_count = component_count,
    df_diagnostic = df_value
  )
}

evaluate_fitted_path <- function(
    fit_object,
    X_test,
    y_test,
    training_response_mean,
    parent_map,
    target_size,
    true_support,
    tolerance = 1e-8) {

  selected <- path_point_at_size(
    coefficient_path = fit_object$coefficient_path,
    parent_map = parent_map,
    target_size = target_size,
    tolerance = tolerance
  )

  intercept_at_step <- fit_object$intercept_path[selected$step]
  predicted <- drop(
    training_response_mean + intercept_at_step + X_test %*% selected$coefficients
  )
  if (!all(is.finite(predicted))) stop("Non-finite test predictions.")
  support <- calculate_support_metrics(selected$active_parents, true_support)
  complexity <- path_point_details(
    coefficient_path = fit_object$coefficient_path,
    parent_map = parent_map,
    target_size = target_size,
    df_diagnostic = fit_object$df_diagnostic,
    tolerance = tolerance
  )

  c(
    support,
    rmse = sqrt(mean((y_test - predicted)^2)),
    selected_parent_count = complexity[["parent_count"]],
    selected_component_count = complexity[["component_count"]],
    df_diagnostic = complexity[["df_diagnostic"]],
    selected_step = complexity[["step"]],
    target_reached = as.integer(selected$selected_size >= target_size),
    target_exact = as.integer(selected$selected_size == target_size)
  )
}

compare_paths_across_representations <- function(
    original_fit,
    sparse_fit,
    sparse_parent_map,
    number_of_parents,
    target_s,
    tolerance = 1e-8) {

  original_parent_map <- seq_len(number_of_parents)
  rank_original <- parent_entry_ranks(
    original_fit$coefficient_path, original_parent_map,
    number_of_parents, tolerance
  )
  rank_sparse <- parent_entry_ranks(
    sparse_fit$coefficient_path, sparse_parent_map,
    number_of_parents, tolerance
  )
  entry_tau <- suppressWarnings(cor(
    rank_original, rank_sparse, method = "kendall", use = "pairwise.complete.obs"
  ))

  get_set <- function(fit, pmap, target) {
    path_point_at_size(fit$coefficient_path, pmap, target, tolerance)$active_parents
  }
  original_s <- get_set(original_fit, original_parent_map, target_s)
  sparse_s <- get_set(sparse_fit, sparse_parent_map, target_s)
  target_2s <- min(2L * target_s, number_of_parents)
  original_2s <- get_set(original_fit, original_parent_map, target_2s)
  sparse_2s <- get_set(sparse_fit, sparse_parent_map, target_2s)

  c(
    entry_order_tau_original_vs_sparse = entry_tau,
    active_set_jaccard_s_original_vs_sparse = jaccard_similarity(original_s, sparse_s),
    active_set_jaccard_2s_original_vs_sparse = jaccard_similarity(original_2s, sparse_2s)
  )
}


## 6. Paired replication and singleton calibration

Each seed regenerates the original dataset. Both methods use its training-only transformations. Group and component results are paired by scenario, replication, seed, scheme, and path.

For the original predictors, singleton group Lasso corresponds to a Lasso objective after accounting for internal RMS standardization. The check compares normalized grids, the expected lambda scaling, and training predictions along the path. F1/RMSE at selected targets and support agreement are also reported; they may differ numerically near entry thresholds. A material prediction or lambda-scale discrepancy stops the run for investigation.


In [7]:
# =============================================================================
# 6. ONE REPLICATION
# =============================================================================

fit_all_controls_for_representation <- function(
    X_train,
    y_train,
    parent_map,
    number_of_parents,
    path_spec) {

  y_mean <- mean(y_train)
  centered_y <- y_train - y_mean

  component_lasso <- extract_component_lasso_control(
    X = X_train,
    centered_y = centered_y,
    nlambda = path_spec$nlambda,
    lambda_min_ratio = path_spec$lambda_min_ratio,
    thresh = path_spec$thresh,
    maxit = path_spec$maxit,
    disable_early_stopping = path_spec$disable_early_stopping,
    type_gaussian = path_spec$type_gaussian
  )

  group_lasso <- list()
  for (weight_spec in GROUP_WEIGHT_SPECIFICATIONS) {
    group_lasso[[weight_spec]] <- fit_group_lasso_controlled(
      X = X_train,
      centered_y = centered_y,
      parent_map = parent_map,
      number_of_parents = number_of_parents,
      nlambda = path_spec$nlambda,
      lambda_min_ratio = path_spec$lambda_min_ratio,
      eps = path_spec$grpreg_eps,
      maxit = path_spec$grpreg_max_iter,
      group_weight_specification = weight_spec
    )
  }

  list(
    training_response_mean = y_mean,
    component_lasso = component_lasso,
    group_lasso = group_lasso
  )
}

representation_row <- function(
    fit_object,
    method,
    group_weight_specification,
    X_test,
    y_test,
    training_response_mean,
    parent_map,
    number_of_parents,
    true_support,
    active_s,
    representation,
    scheme,
    raw_density,
    path_specification) {

  at_s <- evaluate_fitted_path(
    fit_object, X_test, y_test, training_response_mean,
    parent_map, active_s, true_support, COEFFICIENT_TOLERANCE
  )
  target_2s <- min(2L * active_s, number_of_parents)
  at_2s <- evaluate_fitted_path(
    fit_object, X_test, y_test, training_response_mean,
    parent_map, target_2s, true_support, COEFFICIENT_TOLERANCE
  )

  row <- data.frame(
    path_specification = path_specification,
    method = method,
    group_weight_specification = group_weight_specification,
    representation = representation,
    scheme = scheme,
    p_representation = ncol(X_test),
    raw_density = raw_density,
    df_definition = fit_object$df_definition,
    target_parent_count_s = active_s,
    target_parent_count_2s = target_2s,
    target_reached_s = as.logical(at_s[["target_reached"]]),
    target_exact_s = as.logical(at_s[["target_exact"]]),
    target_reached_2s = as.logical(at_2s[["target_reached"]]),
    target_exact_2s = as.logical(at_2s[["target_exact"]]),
    selected_step_s = at_s[["selected_step"]],
    selected_step_2s = at_2s[["selected_step"]],
    group_size_min = min(tabulate(parent_map, nbins = number_of_parents)),
    group_size_max = max(tabulate(parent_map, nbins = number_of_parents)),
    f1_s = unname(at_s["f1"]),
    rmse_s = unname(at_s["rmse"]),
    selected_parent_count_s = unname(at_s["selected_parent_count"]),
    selected_component_count_s = unname(at_s["selected_component_count"]),
    df_diagnostic_s = unname(at_s["df_diagnostic"]),
    f1_2s = unname(at_2s["f1"]),
    rmse_2s = unname(at_2s["rmse"]),
    selected_parent_count_2s = unname(at_2s["selected_parent_count"]),
    selected_component_count_2s = unname(at_2s["selected_component_count"]),
    df_diagnostic_2s = unname(at_2s["df_diagnostic"]),
    returned_path_points = nrow(fit_object$coefficient_path) - 1L,
    stringsAsFactors = FALSE
  )
  diag_row <- as.data.frame(fit_object$diagnostics, stringsAsFactors = FALSE)
  names(diag_row) <- paste0("fit_", names(diag_row))
  cbind(row, diag_row)
}

run_one_penalty_control_replication <- function(
    n, p, correlation, replication, scenario_id,
    schemes, signal_to_noise_ratio, test_sample_size,
    path_spec, base_seed) {

  replication_seed <- as.integer(base_seed + scenario_id * 100000L + replication)
  dat <- generate_regression_data(
    n = n, p = p, correlation = correlation,
    signal_to_noise_ratio = signal_to_noise_ratio,
    test_sample_size = test_sample_size,
    seed = replication_seed
  )

  std <- fit_standardizer(dat$X_train)
  X_train_original <- apply_standardizer(dat$X_train, std)
  X_test_original <- apply_standardizer(dat$X_test, std)

  original_fits <- fit_all_controls_for_representation(
    X_train = X_train_original,
    y_train = dat$y_train,
    parent_map = seq_len(p),
    number_of_parents = p,
    path_spec = path_spec
  )

  representation_rows <- list()
  contrast_rows <- list()
  solver_check_rows <- list()
  row_counter <- 0L
  contrast_counter <- 0L
  solver_check_counter <- 0L

  # Original rows
  row_counter <- row_counter + 1L
  representation_rows[[row_counter]] <- representation_row(
    fit_object = original_fits$component_lasso,
    method = "component_lasso_glmnet",
    group_weight_specification = "not_applicable",
    X_test = X_test_original,
    y_test = dat$y_test,
    training_response_mean = original_fits$training_response_mean,
    parent_map = seq_len(p), number_of_parents = p,
    true_support = dat$true_support, active_s = dat$active_s,
    representation = "original", scheme = "original",
    raw_density = mean(abs(X_train_original) > COEFFICIENT_TOLERANCE),
    path_specification = path_spec$path_specification
  )
  for (weight_spec in GROUP_WEIGHT_SPECIFICATIONS) {
    row_counter <- row_counter + 1L
    representation_rows[[row_counter]] <- representation_row(
      fit_object = original_fits$group_lasso[[weight_spec]],
      method = "parent_group_lasso_grpreg",
      group_weight_specification = weight_spec,
      X_test = X_test_original,
      y_test = dat$y_test,
      training_response_mean = original_fits$training_response_mean,
      parent_map = seq_len(p), number_of_parents = p,
      true_support = dat$true_support, active_s = dat$active_s,
      representation = "original", scheme = "original",
      raw_density = mean(abs(X_train_original) > COEFFICIENT_TOLERANCE),
      path_specification = path_spec$path_specification
    )
  }

  # Solver/calibration check on the original representation. With singleton
  # groups, the group-Lasso penalty reduces to an absolute-value penalty, so
  # this comparison helps distinguish solver/path-calibration differences from
  # the effect of grouping interval components.
  original_component_row <- representation_rows[[1L]]
  for (weight_spec in GROUP_WEIGHT_SPECIFICATIONS) {
    original_group_index <- 1L + match(weight_spec, GROUP_WEIGHT_SPECIFICATIONS)
    original_group_row <- representation_rows[[original_group_index]]
    solver_path_compare <- compare_paths_across_representations(
      original_fits$component_lasso,
      original_fits$group_lasso[[weight_spec]],
      seq_len(p), p, dat$active_s, COEFFICIENT_TOLERANCE
    )
    calibration <- singleton_calibration(
      original_fits$component_lasso, original_fits$group_lasso[[weight_spec]],
      X_train_original, dat$y_train
    )
    solver_check_counter <- solver_check_counter + 1L
    solver_check_rows[[solver_check_counter]] <- data.frame(
      path_specification = path_spec$path_specification,
      group_weight_specification = weight_spec,
      lambda_scale_relative_error = calibration$lambda_scale_relative_error,
      max_training_prediction_rms_over_sd_y = calibration$max_training_prediction_rms_over_sd_y,
      f1_difference_group_minus_glmnet_s = original_group_row$f1_s - original_component_row$f1_s,
      rmse_difference_group_minus_glmnet_s = original_group_row$rmse_s - original_component_row$rmse_s,
      f1_difference_group_minus_glmnet_2s = original_group_row$f1_2s - original_component_row$f1_2s,
      rmse_difference_group_minus_glmnet_2s = original_group_row$rmse_2s - original_component_row$rmse_2s,
      entry_order_tau_glmnet_vs_group = unname(solver_path_compare[1L]),
      active_set_jaccard_s_glmnet_vs_group = unname(solver_path_compare[2L]),
      active_set_jaccard_2s_glmnet_vs_group = unname(solver_path_compare[3L]),
      stringsAsFactors = FALSE
    )
  }

  for (scheme in schemes) {
    sparse <- fit_sparsifier(X_train_original, scheme)
    X_test_sparse <- apply_sparsifier(X_test_original, sparse)
    sparse_fits <- fit_all_controls_for_representation(
      X_train = sparse$X_train,
      y_train = dat$y_train,
      parent_map = sparse$parent_map,
      number_of_parents = p,
      path_spec = path_spec
    )

    # Component-wise Lasso row + contrast
    row_counter <- row_counter + 1L
    sparse_component_row <- representation_row(
      fit_object = sparse_fits$component_lasso,
      method = "component_lasso_glmnet",
      group_weight_specification = "not_applicable",
      X_test = X_test_sparse,
      y_test = dat$y_test,
      training_response_mean = sparse_fits$training_response_mean,
      parent_map = sparse$parent_map, number_of_parents = p,
      true_support = dat$true_support, active_s = dat$active_s,
      representation = "sparsified", scheme = scheme,
      raw_density = sparse$raw_density,
      path_specification = path_spec$path_specification
    )
    representation_rows[[row_counter]] <- sparse_component_row

    original_component_row <- representation_rows[[1L]]
    path_compare <- compare_paths_across_representations(
      original_fits$component_lasso,
      sparse_fits$component_lasso,
      sparse$parent_map, p, dat$active_s, COEFFICIENT_TOLERANCE
    )
    contrast_counter <- contrast_counter + 1L
    contrast_rows[[contrast_counter]] <- data.frame(
      path_specification = path_spec$path_specification,
      method = "component_lasso_glmnet",
      group_weight_specification = "not_applicable",
      scheme = scheme,
      delta_f1_s = sparse_component_row$f1_s - original_component_row$f1_s,
      delta_rmse_s = sparse_component_row$rmse_s - original_component_row$rmse_s,
      delta_f1_2s = sparse_component_row$f1_2s - original_component_row$f1_2s,
      delta_rmse_2s = sparse_component_row$rmse_2s - original_component_row$rmse_2s,
      entry_order_tau_original_vs_sparse = unname(path_compare[1L]),
      active_set_jaccard_s_original_vs_sparse = unname(path_compare[2L]),
      active_set_jaccard_2s_original_vs_sparse = unname(path_compare[3L]),
      original_parent_count_s = original_component_row$selected_parent_count_s,
      sparse_parent_count_s = sparse_component_row$selected_parent_count_s,
      original_component_count_s = original_component_row$selected_component_count_s,
      original_target_reached_s = original_component_row$target_reached_s,
      sparse_target_reached_s = sparse_component_row$target_reached_s,
      original_target_exact_s = original_component_row$target_exact_s,
      sparse_target_exact_s = sparse_component_row$target_exact_s,
      df_definition = sparse_component_row$df_definition,
      sparse_component_count_s = sparse_component_row$selected_component_count_s,
      sparse_df_diagnostic_s = sparse_component_row$df_diagnostic_s,
      stringsAsFactors = FALSE
    )

    # Group-Lasso rows + contrasts
    for (weight_spec in GROUP_WEIGHT_SPECIFICATIONS) {
      row_counter <- row_counter + 1L
      sparse_group_row <- representation_row(
        fit_object = sparse_fits$group_lasso[[weight_spec]],
        method = "parent_group_lasso_grpreg",
        group_weight_specification = weight_spec,
        X_test = X_test_sparse,
        y_test = dat$y_test,
        training_response_mean = sparse_fits$training_response_mean,
        parent_map = sparse$parent_map, number_of_parents = p,
        true_support = dat$true_support, active_s = dat$active_s,
        representation = "sparsified", scheme = scheme,
        raw_density = sparse$raw_density,
        path_specification = path_spec$path_specification
      )
      representation_rows[[row_counter]] <- sparse_group_row

      original_group_index <- 1L + match(weight_spec, GROUP_WEIGHT_SPECIFICATIONS)
      original_group_row <- representation_rows[[original_group_index]]
      path_compare_group <- compare_paths_across_representations(
        original_fits$group_lasso[[weight_spec]],
        sparse_fits$group_lasso[[weight_spec]],
        sparse$parent_map, p, dat$active_s, COEFFICIENT_TOLERANCE
      )
      contrast_counter <- contrast_counter + 1L
      contrast_rows[[contrast_counter]] <- data.frame(
        path_specification = path_spec$path_specification,
        method = "parent_group_lasso_grpreg",
        group_weight_specification = weight_spec,
        scheme = scheme,
        delta_f1_s = sparse_group_row$f1_s - original_group_row$f1_s,
        delta_rmse_s = sparse_group_row$rmse_s - original_group_row$rmse_s,
        delta_f1_2s = sparse_group_row$f1_2s - original_group_row$f1_2s,
        delta_rmse_2s = sparse_group_row$rmse_2s - original_group_row$rmse_2s,
        entry_order_tau_original_vs_sparse = unname(path_compare_group[1L]),
        active_set_jaccard_s_original_vs_sparse = unname(path_compare_group[2L]),
        active_set_jaccard_2s_original_vs_sparse = unname(path_compare_group[3L]),
        original_parent_count_s = original_group_row$selected_parent_count_s,
        sparse_parent_count_s = sparse_group_row$selected_parent_count_s,
        original_component_count_s = original_group_row$selected_component_count_s,
        original_target_reached_s = original_group_row$target_reached_s,
        sparse_target_reached_s = sparse_group_row$target_reached_s,
        original_target_exact_s = original_group_row$target_exact_s,
        sparse_target_exact_s = sparse_group_row$target_exact_s,
        df_definition = sparse_group_row$df_definition,
        sparse_component_count_s = sparse_group_row$selected_component_count_s,
        sparse_df_diagnostic_s = sparse_group_row$df_diagnostic_s,
        stringsAsFactors = FALSE
      )
    }
  }

  rep_df <- do.call(rbind, representation_rows)
  con_df <- do.call(rbind, contrast_rows)
  solver_df <- do.call(rbind, solver_check_rows)
  metadata <- data.frame(
    scenario_id = scenario_id, replication = replication, seed = replication_seed,
    n = n, p = p, dimension = paste0("n=", n, ", p=", p),
    correlation = correlation, active_s = dat$active_s,
    snr = signal_to_noise_ratio, noise_sd = dat$noise_sd,
    stringsAsFactors = FALSE
  )
  rep_df <- cbind(metadata[rep(1L, nrow(rep_df)), ], rep_df)
  con_df <- cbind(metadata[rep(1L, nrow(con_df)), ], con_df)
  solver_df <- cbind(metadata[rep(1L, nrow(solver_df)), ], solver_df)
  rownames(rep_df) <- NULL
  rownames(con_df) <- NULL
  rownames(solver_df) <- NULL
  list(representation = rep_df, contrast = con_df, solver_check = solver_df)
}


## 7. Mandatory preflight: reference integrity and runtime validation

This section checks the complete archived CSV, tests metric extraction, and runs low-dimensional independent and high-dimensional correlated cases on both paths. It compares component-Lasso F1, RMSE, selected parent counts, and data fingerprints against the archived values. Empty joins, duplicate keys, missing rows, or non-finite values fail.

The archive has one known incomplete glmnet path: dense/wide path, scenario 3, replication 37, equal-width four-bin expansion, seed 20560748. It returned 385/400 points with `jerr=-386`. A separate check reruns it with documented iteration-budget retries. The original CSV is left intact; the new run must obtain a complete path and verify the metrics that can be compared. No assumption is made that the unpublished tail leaves all path metrics unchanged.

Use the original R/package environment if reproduction fails. Do not simply loosen a threshold or disable the check.


In [8]:
# 7. Reference checks and preflight. These checks are required in both modes.

key_string <- function(d, keys) {
  if (!all(keys %in% names(d))) stop("Missing key columns.")
  if (anyNA(d[keys])) stop("Missing values in comparison keys.")
  do.call(paste, c(d[keys], sep = "\t"))
}

hash_object <- function(object) {
  tmp <- tempfile(fileext = ".rds")
  on.exit(unlink(tmp), add = TRUE)
  saveRDS(object, tmp, version = 2)
  unname(tools::md5sum(tmp))
}

read_baseline <- function(filename) {
  if (!file.exists(filename)) stop("Required reference CSV missing: ", filename,
                                  ". Run from the extracted original project directory.")
  d <- read.csv(filename, stringsAsFactors = FALSE)
  required <- c(REFERENCE_KEYS, "noise_sd", "p_representation", "raw_density",
                "f1_glmnet", "rmse_glmnet", "selected_size_glmnet", "selected_size_2s_glmnet",
                "glmnet_requested_lambda_points", "glmnet_returned_lambda_points",
                "glmnet_requested_lambda_max", "glmnet_requested_lambda_min_ratio",
                "glmnet_complete_requested_grid", "glmnet_jerr")
  if (!all(required %in% names(d))) stop("Reference CSV is missing required columns.")
  if (nrow(d) != 16000L || anyDuplicated(key_string(d, REFERENCE_KEYS))) {
    stop("Expected the complete original 16,000-row CSV with unique keys.")
  }
  numeric_fields <- setdiff(required, c(REFERENCE_KEYS, "glmnet_complete_requested_grid"))
  if (!all(vapply(d[numeric_fields], function(x) is.numeric(x) && all(is.finite(x)), logical(1)))) {
    stop("Reference has missing/non-finite numeric values.")
  }
  expected_design <- merge(canonical_design[c("scenario_id", "n", "p", "correlation")],
                           expand.grid(replication = seq_len(100L),
                                       path_specification = c("primary_controlled", "sensitivity_dense_wide"),
                                       scheme = c("original", SPARSIFICATION_SCHEMES),
                                       stringsAsFactors = FALSE), by = NULL)
  expected_design$representation <- ifelse(expected_design$scheme == "original", "original", "sparsified")
  expected_design$seed <- as.integer(BASE_SEED + expected_design$scenario_id * 100000L + expected_design$replication)
  if (!setequal(key_string(d, REFERENCE_KEYS), key_string(expected_design, REFERENCE_KEYS))) {
    stop("Reference scenario/seed/path coverage differs from the supplied original design.")
  }
  d
}

validate_representation_rows <- function(d) {
  fields <- c("f1_s", "rmse_s", "f1_2s", "rmse_2s",
              "selected_parent_count_s", "selected_component_count_s", "df_diagnostic_s",
              "selected_parent_count_2s", "selected_component_count_2s", "df_diagnostic_2s",
              "selected_step_s", "selected_step_2s")
  stopifnot(nrow(d) > 0L, all(fields %in% names(d)),
            all(vapply(d[fields], function(x) all(is.finite(x)), logical(1))),
            all(d$fit_accepted_path), all(d$fit_complete_requested_grid),
            all(d$returned_path_points == d$fit_requested_lambda_points))
  for (suffix in c("s", "2s")) {
    parents <- d[[paste0("selected_parent_count_", suffix)]]
    components <- d[[paste0("selected_component_count_", suffix)]]
    target <- d[[paste0("target_parent_count_", suffix)]]
    step <- d[[paste0("selected_step_", suffix)]]
    f1 <- d[[paste0("f1_", suffix)]]
    stopifnot(all(parents == floor(parents)), all(components == floor(components)),
              all(parents >= 0 & parents <= d$p), all(components >= parents),
              all(components <= d$p_representation), all(f1 >= 0 & f1 <= 1),
              all(d[[paste0("rmse_", suffix)]] >= 0),
              all(step >= 1 & step <= d$returned_path_points + 1),
              all(d[[paste0("target_reached_", suffix)]] == (parents >= target)),
              all(d[[paste0("target_exact_", suffix)]] == (parents == target)))
  }
  invisible(TRUE)
}

compare_reference <- function(current, stage, write_files = TRUE) {
  current <- current[current$method == "component_lasso_glmnet", , drop = FALSE]
  if (nrow(current) == 0L) stop("Empty baseline reproduction comparison.")
  validate_representation_rows(current)
  current_keys <- key_string(current, REFERENCE_KEYS)
  if (anyDuplicated(current_keys)) stop("Duplicate current reference keys.")
  reference_keys <- key_string(reference_results, REFERENCE_KEYS)
  if (anyDuplicated(reference_keys)) stop("Duplicate archived reference keys.")
  index <- match(current_keys, reference_keys)
  if (anyNA(index)) stop("Some current rows have no exact archived reference match.")
  old <- reference_results[index, , drop = FALSE]
  check <- current[REFERENCE_KEYS]
  check$abs_f1_difference <- abs(current$f1_s - old$f1_glmnet)
  check$abs_rmse_difference <- abs(current$rmse_s - old$rmse_glmnet)
  check$abs_parent_count_difference_s <- abs(current$selected_parent_count_s - old$selected_size_glmnet)
  check$abs_parent_count_difference_2s <- abs(current$selected_parent_count_2s - old$selected_size_2s_glmnet)
  check$relative_noise_sd_difference <- abs(current$noise_sd - old$noise_sd) / pmax(1, abs(old$noise_sd))
  check$component_dimension_difference <- abs(current$p_representation - old$p_representation)
  check$raw_density_difference <- abs(current$raw_density - old$raw_density)
  check$lambda_max_relative_difference <- abs(current$fit_returned_lambda_max / old$glmnet_requested_lambda_max - 1)
  check$requested_points_difference <- abs(current$fit_requested_lambda_points - old$glmnet_requested_lambda_points)
  check$requested_ratio_difference <- abs(current$fit_requested_lambda_min_ratio - old$glmnet_requested_lambda_min_ratio)
  difference_names <- setdiff(names(check), REFERENCE_KEYS)
  if (!all(vapply(check[difference_names], function(x) all(is.finite(x)), logical(1)))) {
    stop("Non-finite baseline differences; reproduction has not been established.")
  }
  limits <- c(REFERENCE_F1_TOLERANCE, REFERENCE_RMSE_TOLERANCE, 0, 0,
              1e-10, 0, 1e-12, 1e-9, 0, 1e-12)
  maxima <- vapply(check[difference_names], max, numeric(1))
  report <- data.frame(stage = stage, matched_rows = nrow(check),
                       expected_current_rows = nrow(current),
                       archived_partial_path_rows = sum(!old$glmnet_complete_requested_grid | old$glmnet_jerr != 0),
                       current_retried_rows = sum(current$fit_attempt_count > 1L),
                       all_checks_passed = all(maxima <= limits),
                       stringsAsFactors = FALSE)
  for (name in names(maxima)) report[[paste0("max_", name)]] <- maxima[[name]]
  check$archived_returned_points <- old$glmnet_returned_lambda_points
  check$current_returned_points <- current$returned_path_points
  check$archived_jerr <- old$glmnet_jerr
  if (write_files) {
    write.csv(check, output_path(paste0(stage, "_baseline_details.csv")), row.names = FALSE)
    write.csv(report, output_path(paste0(stage, "_baseline_report.csv")), row.names = FALSE)
  }
  if (!isTRUE(report$all_checks_passed)) {
    stop("Baseline reproduction failed in ", stage, ". Check the saved differences and original R/package environment.")
  }
  report
}

run_with_failure_log <- function(context, expression) {
  tryCatch(force(expression), error = function(e) {
    filename <- tempfile(pattern = "failure_", tmpdir = OUTPUT_DIRECTORY, fileext = ".rds")
    details <- list(context = context, message = conditionMessage(e),
                    attempts = e$attempts, signature = RUN_SIGNATURE,
                    time = as.character(Sys.time()))
    saveRDS(details, filename)
    capture.output(str(details), file = sub("\\.rds$", ".txt", filename))
    stop(conditionMessage(e), "\nFailure details: ", filename, call. = FALSE)
  })
}

test_metric_extraction <- function() {
  mock <- list(coefficient_path = rbind(c(0,0,0,0), c(1,0,0,0),
                                        c(1,2,0,0), c(1,2,3,0)),
               intercept_path = rep(0, 4), df_diagnostic = c(null=0,s1=1,s2=2,s3=3))
  z <- evaluate_fitted_path(mock, diag(4), c(1,2,3,0), 0,
                            c(1L,1L,2L,3L), 2L, c(1L,2L))
  stopifnot(all(is.finite(z)), z[["f1"]] == 1, z[["rmse"]] == 0,
            z[["selected_parent_count"]] == 2,
            z[["selected_component_count"]] == 3,
            z[["df_diagnostic"]] == 3, z[["selected_step"]] == 4)
  fallback <- evaluate_fitted_path(mock, diag(4), c(1,2,3,0), 0,
                                   c(1L,1L,2L,3L), 3L, c(1L,2L))
  stopifnot(fallback[["target_reached"]] == 0,
            fallback[["target_exact"]] == 0,
            fallback[["selected_parent_count"]] == 2)
  invisible(TRUE)
}

# Helpers are placed before preflight so their code enters the run signature.
checkpoint_path <- function(path_specification, scenario_id, replication) {
  folder <- file.path(CHECKPOINT_DIRECTORY, path_specification,
                      sprintf("scenario_%02d", scenario_id))
  dir.create(folder, recursive = TRUE, showWarnings = FALSE)
  file.path(folder, sprintf("replication_%03d.rds", replication))
}

validate_replication <- function(ans, scenario_row, path_spec, replication) {
  rep <- ans$representation
  validate_representation_rows(rep)
  methods_per_representation <- 1L + length(GROUP_WEIGHT_SPECIFICATIONS)
  stopifnot(nrow(rep) == (1L + length(SPARSIFICATION_SCHEMES)) * methods_per_representation,
            nrow(ans$contrast) == length(SPARSIFICATION_SCHEMES) * methods_per_representation,
            nrow(ans$solver_check) == length(GROUP_WEIGHT_SPECIFICATIONS),
            all(rep$scenario_id == scenario_row$scenario_id),
            all(rep$replication == replication),
            all(rep$path_specification == path_spec$path_specification),
            all(rep$seed == as.integer(BASE_SEED + scenario_row$scenario_id * 100000L + replication)),
            !anyDuplicated(rep[c("method", "group_weight_specification", "representation", "scheme")]))
  invisible(TRUE)
}

atomic_checkpoint <- function(value, filename) {
  # Each replication has its own immutable file: no overwrite/rename ambiguity
  # on Windows. A temporary file is renamed only after saveRDS succeeds.
  if (file.exists(filename)) stop("Checkpoint already exists: ", filename)
  tmp <- tempfile(pattern = "pending_", tmpdir = dirname(filename), fileext = ".rds")
  on.exit(if (file.exists(tmp)) unlink(tmp), add = TRUE)
  saveRDS(value, tmp)
  if (!file.rename(tmp, filename)) stop("Could not finalize checkpoint: ", filename)
}

run_penalty_control_scenario <- function(scenario_row, path_spec) {
  completed <- vector("list", NUMBER_OF_REPLICATIONS)
  for (replication in seq_len(NUMBER_OF_REPLICATIONS)) {
    cp <- checkpoint_path(path_spec$path_specification, scenario_row$scenario_id, replication)
    cache_key <- paste(path_spec$path_specification, scenario_row$scenario_id, replication, sep = "_")
    if (file.exists(cp)) {
      saved <- readRDS(cp)
      if (!identical(saved$signature, RUN_SIGNATURE)) stop("Checkpoint signature mismatch: ", cp)
      ans <- saved$result
    } else {
      context <- list(stage = "experiment", scenario_id = scenario_row$scenario_id,
                      replication = replication, path = path_spec$path_specification,
                      seed = as.integer(BASE_SEED + scenario_row$scenario_id * 100000L + replication))
      cat(path_spec$path_specification, "scenario", scenario_row$scenario_id,
          "replication", replication, "/", NUMBER_OF_REPLICATIONS, "\n")
      if (!is.null(preflight_cache[[cache_key]])) {
        ans <- preflight_cache[[cache_key]]
      } else {
        ans <- run_with_failure_log(context, run_one_penalty_control_replication(
          n = scenario_row$n, p = scenario_row$p, correlation = scenario_row$correlation,
          replication = replication, scenario_id = scenario_row$scenario_id,
          schemes = SPARSIFICATION_SCHEMES, signal_to_noise_ratio = SIGNAL_TO_NOISE_RATIO,
          test_sample_size = TEST_SAMPLE_SIZE, path_spec = path_spec, base_seed = BASE_SEED
        ))
      }
      validate_replication(ans, scenario_row, path_spec, replication)
      # Verify every baseline row before committing a replication checkpoint.
      compare_reference(ans$representation, "checkpoint_validation", write_files = FALSE)
      atomic_checkpoint(list(signature = RUN_SIGNATURE, result = ans), cp)
    }
    validate_replication(ans, scenario_row, path_spec, replication)
    completed[[replication]] <- ans
  }
  completed
}



# Initialize only after functions are defined. Checkpoint signatures include
# function bodies/formals (not their mutable global environment).
REFERENCE_KEYS <- c("path_specification", "scenario_id", "replication", "seed",
                    "n", "p", "correlation", "representation", "scheme")
reference_results <- read_baseline(REFERENCE_FILE)
analysis_names <- Filter(function(name) is.function(get(name, envir = .GlobalEnv)),
                         ls(envir = .GlobalEnv))
function_signature <- lapply(sort(analysis_names), function(name) {
  fun <- get(name, envir = .GlobalEnv)
  list(name = name, formals = formals(fun), body = deparse(body(fun)))
})
run_configuration <- list(
  revision = NOTEBOOK_VERSION, mode = RUN_MODE, replications = NUMBER_OF_REPLICATIONS,
  seed = BASE_SEED, snr = SIGNAL_TO_NOISE_RATIO, test_n = TEST_SAMPLE_SIZE,
  tolerance = COEFFICIENT_TOLERANCE, design = simulation_design,
  schemes = SPARSIFICATION_SCHEMES, weights = GROUP_WEIGHT_SPECIFICATIONS,
  paths = PATH_SPECIFICATIONS, retries = ITERATION_BUDGET_MULTIPLIERS,
  reference_f1_tolerance = REFERENCE_F1_TOLERANCE,
  reference_rmse_tolerance = REFERENCE_RMSE_TOLERANCE,
  singleton_prediction_tolerance = SINGLETON_PREDICTION_TOLERANCE,
  reference_md5 = unname(tools::md5sum(REFERENCE_FILE)),
  functions = function_signature, R_version = R.version.string,
  RNG = RNGkind(), packages = vapply(c(required_packages, "Matrix"), function(x)
    as.character(packageVersion(x)), character(1))
)
RUN_SIGNATURE <- hash_object(run_configuration)
OUTPUT_DIRECTORY <- file.path(OUTPUT_ROOT, RUN_MODE, RUN_SIGNATURE)
CHECKPOINT_DIRECTORY <- file.path(OUTPUT_DIRECTORY, "checkpoints")
dir.create(CHECKPOINT_DIRECTORY, recursive = TRUE, showWarnings = FALSE)
saveRDS(run_configuration, output_path("run_configuration.rds"))
capture.output(sessionInfo(), file = output_path("session_info.txt"))
write.csv(PATH_SPECIFICATIONS, output_path("path_specifications.csv"), row.names = FALSE)
write.csv(simulation_design, output_path("simulation_design.csv"), row.names = FALSE)
write.csv(data.frame(group_weight_specification = GROUP_WEIGHT_SPECIFICATIONS),
          output_path("group_weight_specifications.csv"), row.names = FALSE)
writeLines(RUN_SIGNATURE, output_path("run_signature.txt"))
cat("Output directory:", normalizePath(OUTPUT_DIRECTORY), "\n")

reference_path_issues <- reference_results[
  !reference_results$glmnet_complete_requested_grid | reference_results$glmnet_jerr != 0, ]
write.csv(reference_path_issues, output_path("archived_glmnet_path_issues.csv"), row.names = FALSE)
cat("Archived incomplete/error paths:", nrow(reference_path_issues), "of", nrow(reference_results), "\n")
test_metric_extraction()

# Always execute these exact-data checks, including in full mode.
# Cached preflight replications are reused by the main runner.
preflight_cache <- list()
preflight_reports <- list()
preflight_design <- canonical_design[canonical_design$scenario_id %in% c(1L, 11L), , drop = FALSE]
for (path_index in seq_len(nrow(PATH_SPECIFICATIONS))) {
  spec <- PATH_SPECIFICATIONS[path_index, , drop = FALSE]
  for (case in seq_len(nrow(preflight_design))) {
    d <- preflight_design[case, , drop = FALSE]
    context <- list(stage = "preflight", scenario_id = d$scenario_id, replication = 1L,
                    seed = as.integer(BASE_SEED + d$scenario_id * 100000L + 1L),
                    path = spec$path_specification)
    ans <- run_with_failure_log(context, run_one_penalty_control_replication(
      n = d$n, p = d$p, correlation = d$correlation, replication = 1L,
      scenario_id = d$scenario_id, schemes = SPARSIFICATION_SCHEMES,
      signal_to_noise_ratio = SIGNAL_TO_NOISE_RATIO, test_sample_size = TEST_SAMPLE_SIZE,
      path_spec = spec, base_seed = BASE_SEED
    ))
    validate_representation_rows(ans$representation)
    stage <- paste0("preflight_", spec$path_specification, "_scenario_", d$scenario_id)
    preflight_reports[[stage]] <- compare_reference(ans$representation, stage)
    preflight_cache[[paste(spec$path_specification, d$scenario_id, 1L, sep = "_")]] <- ans
    write.csv(ans$representation, output_path(paste0(stage, "_fits.csv")), row.names = FALSE)
    cat("Preflight passed:", stage, "\n")
  }
}

# Verify that deliberately empty, missing, and duplicate comparisons cannot pass.
valid_example <- preflight_cache[[1L]]$representation
expect_reference_failure <- function(d) {
  failed <- inherits(try(compare_reference(d, "intentional_failure", write_files = FALSE), silent = TRUE), "try-error")
  if (!failed) stop("Reproduction guard accepted an intentionally invalid input.")
}
expect_reference_failure(valid_example[FALSE, , drop = FALSE])
bad_example <- valid_example
bad_example$selected_parent_count_s[bad_example$method == "component_lasso_glmnet"] <- NA_real_
expect_reference_failure(bad_example)
expect_reference_failure(rbind(valid_example, valid_example))

# Explicitly exercise the single archived truncated path; this remains separate
# from the Monte Carlo sample and is never counted as an extra replication.
if ("sensitivity_dense_wide" %in% PATH_SPECIFICATIONS$path_specification) {
  spec <- PATH_SPECIFICATIONS[PATH_SPECIFICATIONS$path_specification == "sensitivity_dense_wide", , drop = FALSE]
  known_seed <- as.integer(BASE_SEED + 3L * 100000L + 37L)
  known <- run_with_failure_log(list(stage = "archived_truncated_path", scenario_id = 3L,
                                    replication = 37L, seed = known_seed), {
    dat <- generate_regression_data(100L, 20L, "ar1_rho_0.9", SIGNAL_TO_NOISE_RATIO,
                                    TEST_SAMPLE_SIZE, known_seed)
    standardizer <- fit_standardizer(dat$X_train)
    train <- apply_standardizer(dat$X_train, standardizer)
    test <- apply_standardizer(dat$X_test, standardizer)
    sparse <- fit_sparsifier(train, "equal_width_4")
    sparse_test <- apply_sparsifier(test, sparse)
    fit <- extract_component_lasso_control(
      sparse$X_train, dat$y_train - mean(dat$y_train), spec$nlambda,
      spec$lambda_min_ratio, spec$thresh, spec$maxit,
      spec$disable_early_stopping, spec$type_gaussian)
    row <- representation_row(fit, "component_lasso_glmnet", "not_applicable",
                               sparse_test, dat$y_test, mean(dat$y_train),
                               sparse$parent_map, 20L, dat$true_support, dat$active_s,
                               "sparsified", "equal_width_4", sparse$raw_density,
                               spec$path_specification)
    cbind(data.frame(scenario_id = 3L, replication = 37L, seed = known_seed,
                      n = 100L, p = 20L, correlation = "ar1_rho_0.9",
                      noise_sd = dat$noise_sd), row)
  })
  write.csv(known, output_path("archived_truncated_path_rerun.csv"), row.names = FALSE)
  preflight_reports[["archived_truncated_path"]] <- compare_reference(known, "archived_truncated_path")
}
write.csv(do.call(rbind, preflight_reports), output_path("preflight_baseline_reports.csv"), row.names = FALSE)
cat("Required preflight checks passed.\n")


Output directory: C:\Users\it08d\OneDrive - kcg.ac.jp\_KCGI\_Projects\_Done\Xia_sparsification\sparsification\outputs\parent_group_penalty_sensitivity\full\c069d39e6758abb3a27e004cee7866a8 
Archived incomplete/error paths: 1 of 16000 
Preflight passed: preflight_primary_controlled_scenario_1 
Preflight passed: preflight_primary_controlled_scenario_11 
Preflight passed: preflight_sensitivity_dense_wide_scenario_1 
Preflight passed: preflight_sensitivity_dense_wide_scenario_11 
Required preflight checks passed.


## 8. Run with configuration-specific checkpoints

Outputs are separated by quick/full mode and a signature of analysis functions, settings, baseline CSV, R version, and package versions. Each successful replication has an immutable checkpoint written through a temporary file and rename. A failed replication is logged with its seed; it is not added to the result tables. Complete coverage and baseline reproduction are required before summaries are made.


In [ ]:
# 8. Execute or load this configuration.
all_results <- list()
if (RUN_EXPERIMENT) {
  result_counter <- 0L
  for (path_index in seq_len(nrow(PATH_SPECIFICATIONS))) {
    path_spec <- PATH_SPECIFICATIONS[path_index, , drop = FALSE]
    for (scenario_index in seq_len(nrow(simulation_design))) {
      scenario_row <- simulation_design[scenario_index, , drop = FALSE]
      scenario_results <- run_penalty_control_scenario(scenario_row, path_spec)
      for (ans in scenario_results) {
        result_counter <- result_counter + 1L
        all_results[[result_counter]] <- ans
      }
    }
  }
  representation_results <- do.call(rbind, lapply(all_results, `[[`, "representation"))
  contrast_results <- do.call(rbind, lapply(all_results, `[[`, "contrast"))
  solver_check_results <- do.call(rbind, lapply(all_results, `[[`, "solver_check"))
  rownames(representation_results) <- rownames(contrast_results) <- rownames(solver_check_results) <- NULL
} else {
  representation_results <- read.csv(output_path("representation_level_results.csv"))
  contrast_results <- read.csv(output_path("replication_level_contrasts.csv"))
  solver_check_results <- read.csv(output_path("original_singleton_solver_check.csv"))
}

validate_representation_rows(representation_results)
base_reps <- nrow(PATH_SPECIFICATIONS) * nrow(simulation_design) * NUMBER_OF_REPLICATIONS
methods_per_representation <- 1L + length(GROUP_WEIGHT_SPECIFICATIONS)
stopifnot(nrow(representation_results) == base_reps * (1L + length(SPARSIFICATION_SCHEMES)) * methods_per_representation,
          nrow(contrast_results) == base_reps * length(SPARSIFICATION_SCHEMES) * methods_per_representation,
          nrow(solver_check_results) == base_reps * length(GROUP_WEIGHT_SPECIFICATIONS),
          !anyDuplicated(representation_results[c(REFERENCE_KEYS, "method", "group_weight_specification")]))
expected_reps <- expand.grid(path_specification = PATH_SPECIFICATIONS$path_specification,
                             scenario_id = simulation_design$scenario_id,
                             replication = seq_len(NUMBER_OF_REPLICATIONS),
                             stringsAsFactors = FALSE)
stopifnot(setequal(key_string(unique(representation_results[c("path_specification", "scenario_id", "replication")]),
                                c("path_specification", "scenario_id", "replication")),
                    key_string(expected_reps, c("path_specification", "scenario_id", "replication"))))
reproduction_report <- compare_reference(representation_results, "complete_run")
write.csv(representation_results, output_path("representation_level_results.csv"), row.names = FALSE)
write.csv(contrast_results, output_path("replication_level_contrasts.csv"), row.names = FALSE)
write.csv(solver_check_results, output_path("original_singleton_solver_check.csv"), row.names = FALSE)
cat("Validated complete coverage and baseline reproduction before summaries.\n")


## 9. Configuration summaries

Full mode estimates mean paired changes and approximate 95% Monte Carlo intervals across 100 replications. Quick mode is a validation run with two replications; its intervals are not intended for substantive interpretation. Sample counts are exported for every metric. The difference between the two paired changes measures sensitivity to the two fitted penalty constructions; it is not a causal decomposition of representation and penalty.

The attainment summary reports target-reached and exact-target rates for each method and representation. The separately labeled degrees-of-freedom diagnostics are not comparable estimates of a common exact quantity.


In [ ]:
# =============================================================================
# 9. SUMMARIES
# =============================================================================

mc_summary <- function(x) {
  x <- x[is.finite(x)]
  n <- length(x)
  if (n == 0L) return(c(mean = NA, sd = NA, n = 0, ci_low = NA, ci_high = NA))
  m <- mean(x)
  s <- if (n > 1L) sd(x) else NA_real_
  se <- s / sqrt(n)
  c(mean = m, sd = s, n = n, ci_low = m - 1.96 * se, ci_high = m + 1.96 * se)
}

summary_metrics <- c(
  "delta_f1_s", "delta_rmse_s", "delta_f1_2s", "delta_rmse_2s",
  "entry_order_tau_original_vs_sparse",
  "active_set_jaccard_s_original_vs_sparse",
  "active_set_jaccard_2s_original_vs_sparse",
  "original_parent_count_s", "sparse_parent_count_s",
  "original_component_count_s", "sparse_component_count_s", "sparse_df_diagnostic_s",
  "original_target_reached_s", "sparse_target_reached_s",
  "original_target_exact_s", "sparse_target_exact_s"
)

group_keys <- c(
  "path_specification", "n", "p", "dimension", "correlation", "active_s",
  "scheme", "method", "group_weight_specification", "df_definition"
)

split_key <- do.call(
  interaction,
  c(contrast_results[group_keys], list(drop = TRUE, lex.order = TRUE))
)
summary_list <- lapply(split(contrast_results, split_key), function(d) {
  base <- d[1L, group_keys, drop = FALSE]
  out <- base
  for (metric in summary_metrics) {
    z <- mc_summary(d[[metric]])
    out[[paste0(metric, "_mean")]] <- unname(z["mean"])
    out[[paste0(metric, "_sd")]] <- unname(z["sd"])
    out[[paste0(metric, "_n")]] <- unname(z["n"])
    out[[paste0(metric, "_ci_low")]] <- unname(z["ci_low"])
    out[[paste0(metric, "_ci_high")]] <- unname(z["ci_high"])
  }
  out
})
configuration_summary <- do.call(rbind, summary_list)
rownames(configuration_summary) <- NULL
write.csv(configuration_summary, output_path("configuration_summary.csv"), row.names = FALSE)

# Penalty-sensitivity contrast: compare sparse-minus-original effects between
# component-wise glmnet Lasso and equal-parent group Lasso within each exact
# replication/configuration.
component <- contrast_results[
  contrast_results$method == "component_lasso_glmnet", , drop = FALSE
]
group_equal <- contrast_results[
  contrast_results$method == "parent_group_lasso_grpreg" &
    contrast_results$group_weight_specification == "equal_parent", , drop = FALSE
]

merge_keys <- c(
  "path_specification", "scenario_id", "replication", "seed", "n", "p",
  "dimension", "correlation", "active_s", "scheme"
)
stopifnot(nrow(component) > 0L, nrow(component) == nrow(group_equal),
          !anyDuplicated(component[merge_keys]), !anyDuplicated(group_equal[merge_keys]))
paired_penalty <- merge(
  component, group_equal,
  by = merge_keys,
  suffixes = c("_component_lasso", "_group_lasso"),
  all = FALSE
)

stopifnot(nrow(paired_penalty) == nrow(component))
for (metric in c("delta_f1_s", "delta_rmse_s", "delta_f1_2s", "delta_rmse_2s")) {
  paired_penalty[[paste0("penalty_sensitivity_", metric)]] <-
    paired_penalty[[paste0(metric, "_group_lasso")]] -
    paired_penalty[[paste0(metric, "_component_lasso")]]
}
paired_penalty$penalty_sensitivity_entry_tau <-
  paired_penalty$entry_order_tau_original_vs_sparse_group_lasso -
  paired_penalty$entry_order_tau_original_vs_sparse_component_lasso

write.csv(paired_penalty, output_path("penalty_sensitivity_replication_level.csv"), row.names = FALSE)

penalty_metrics <- c(
  "penalty_sensitivity_delta_f1_s",
  "penalty_sensitivity_delta_rmse_s",
  "penalty_sensitivity_delta_f1_2s",
  "penalty_sensitivity_delta_rmse_2s",
  "penalty_sensitivity_entry_tau"
)
penalty_group_keys <- c(
  "path_specification", "n", "p", "dimension", "correlation", "active_s", "scheme"
)
penalty_key <- do.call(
  interaction,
  c(paired_penalty[penalty_group_keys], list(drop = TRUE, lex.order = TRUE))
)
penalty_summary_list <- lapply(split(paired_penalty, penalty_key), function(d) {
  out <- d[1L, penalty_group_keys, drop = FALSE]
  for (metric in penalty_metrics) {
    z <- mc_summary(d[[metric]])
    out[[paste0(metric, "_mean")]] <- unname(z["mean"])
    out[[paste0(metric, "_n")]] <- unname(z["n"])
    out[[paste0(metric, "_ci_low")]] <- unname(z["ci_low"])
    out[[paste0(metric, "_ci_high")]] <- unname(z["ci_high"])
  }
  out
})
penalty_sensitivity_summary <- do.call(rbind, penalty_summary_list)
rownames(penalty_sensitivity_summary) <- NULL
write.csv(
  penalty_sensitivity_summary,
  output_path("penalty_sensitivity_configuration_summary.csv"),
  row.names = FALSE
)

print(head(configuration_summary, 12))
print(head(penalty_sensitivity_summary, 12))

# Original-representation singleton-group solver/calibration summary.
solver_metrics <- c(
  "lambda_scale_relative_error", "max_training_prediction_rms_over_sd_y",
  "f1_difference_group_minus_glmnet_s",
  "rmse_difference_group_minus_glmnet_s",
  "entry_order_tau_glmnet_vs_group",
  "active_set_jaccard_s_glmnet_vs_group"
)
solver_keys <- c(
  "path_specification", "n", "p", "dimension", "correlation",
  "active_s", "group_weight_specification"
)
solver_key <- do.call(
  interaction,
  c(solver_check_results[solver_keys], list(drop = TRUE, lex.order = TRUE))
)
solver_summary_list <- lapply(split(solver_check_results, solver_key), function(d) {
  out <- d[1L, solver_keys, drop = FALSE]
  for (metric in solver_metrics) {
    z <- mc_summary(d[[metric]])
    out[[paste0(metric, "_mean")]] <- unname(z["mean"])
    out[[paste0(metric, "_n")]] <- unname(z["n"])
    out[[paste0(metric, "_ci_low")]] <- unname(z["ci_low"])
    out[[paste0(metric, "_ci_high")]] <- unname(z["ci_high"])
  }
  out
})
solver_check_summary <- do.call(rbind, solver_summary_list)
rownames(solver_check_summary) <- NULL
write.csv(
  solver_check_summary,
  output_path("original_singleton_solver_check_summary.csv"),
  row.names = FALSE
)
cat("Original singleton-group solver check summary:\n")
print(head(solver_check_summary, 12))

attainment_keys <- c("path_specification", "scenario_id", "representation", "scheme",
                    "method", "group_weight_specification", "df_definition")
attainment_split <- do.call(interaction, c(representation_results[attainment_keys],
                                          list(drop = TRUE, lex.order = TRUE)))
attainment_summary <- do.call(rbind, lapply(split(representation_results, attainment_split), function(d) {
  z <- d[1L, attainment_keys, drop = FALSE]
  z$replications <- nrow(d)
  for (metric in c("target_reached_s", "target_exact_s", "target_reached_2s", "target_exact_2s",
                   "selected_parent_count_s", "selected_component_count_s",
                   "selected_parent_count_2s", "selected_component_count_2s")) {
    z[[paste0(metric, "_mean")]] <- mean(d[[metric]])
  }
  z
}))
write.csv(attainment_summary, output_path("target_attainment.csv"), row.names = FALSE)


   path_specification   n  p   dimension correlation active_s        scheme
1  primary_controlled 100 20 n=100, p=20 independent        5 equal_width_4
2  primary_controlled 100 20 n=100, p=20 independent        5 equal_width_4
3  primary_controlled 100 20 n=100, p=20 independent        5 equal_width_4
4  primary_controlled 100 20 n=100, p=20 independent        5    quantile_2
5  primary_controlled 100 20 n=100, p=20 independent        5    quantile_2
6  primary_controlled 100 20 n=100, p=20 independent        5    quantile_2
7  primary_controlled 100 20 n=100, p=20 independent        5    quantile_4
8  primary_controlled 100 20 n=100, p=20 independent        5    quantile_4
9  primary_controlled 100 20 n=100, p=20 independent        5    quantile_4
10 primary_controlled 100 20 n=100, p=20 independent        5    quantile_8
11 primary_controlled 100 20 n=100, p=20 independent        5    quantile_8
12 primary_controlled 100 20 n=100, p=20 independent        5    quantile_8
            

## 10. Descriptive aggregate diagnostics

Configuration sign counts and medians below are descriptive, not significance tests. Examine the complete configuration summaries and paired uncertainty intervals before drawing conclusions. A direction that persists can be described as persisting under the examined parent-group penalty. It does not establish that the original effect was independent of penalty geometry or model complexity. Quick-mode tables and figures are for validation only.


In [ ]:
# =============================================================================
# 10. AGGREGATE DIAGNOSTICS
# =============================================================================

primary <- configuration_summary[
  configuration_summary$path_specification == PRIMARY_PATH_SPECIFICATION, , drop = FALSE
]

aggregate_rows <- list()
idx <- 0L
for (method_name in unique(primary$method)) {
  method_data <- primary[primary$method == method_name, , drop = FALSE]
  weight_values <- unique(method_data$group_weight_specification)
  for (weight_spec in weight_values) {
    d <- method_data[method_data$group_weight_specification == weight_spec, , drop = FALSE]
    idx <- idx + 1L
    aggregate_rows[[idx]] <- data.frame(
      method = method_name,
      group_weight_specification = weight_spec,
      configurations = nrow(d),
      configurations_f1_improved = sum(d$delta_f1_s_mean > 0, na.rm = TRUE),
      configurations_f1_decreased = sum(d$delta_f1_s_mean < 0, na.rm = TRUE),
      configurations_rmse_improved = sum(d$delta_rmse_s_mean < 0, na.rm = TRUE),
      configurations_rmse_worsened = sum(d$delta_rmse_s_mean > 0, na.rm = TRUE),
      median_delta_f1_s = median(d$delta_f1_s_mean, na.rm = TRUE),
      median_delta_rmse_s = median(d$delta_rmse_s_mean, na.rm = TRUE),
      median_entry_tau_original_vs_sparse = median(
        d$entry_order_tau_original_vs_sparse_mean, na.rm = TRUE
      ),
      median_jaccard_s_original_vs_sparse = median(
        d$active_set_jaccard_s_original_vs_sparse_mean, na.rm = TRUE
      ),
      median_sparse_component_count_at_s = median(
        d$sparse_component_count_s_mean, na.rm = TRUE
      ),
      stringsAsFactors = FALSE
    )
  }
}
aggregate_diagnostics <- do.call(rbind, aggregate_rows)
write.csv(aggregate_diagnostics, output_path("aggregate_diagnostics.csv"), row.names = FALSE)
print(aggregate_diagnostics)

# Compact primary-control summary: primary path, component Lasso vs
# equal-parent group Lasso only. Complete configuration-level values remain in CSV.
primary_control_summary <- primary[
  primary$method == "component_lasso_glmnet" |
    (primary$method == "parent_group_lasso_grpreg" &
       primary$group_weight_specification == "equal_parent"),
  c(
    "dimension", "correlation", "scheme", "method", "group_weight_specification", "df_definition",
    "delta_f1_s_mean", "delta_f1_s_ci_low", "delta_f1_s_ci_high",
    "delta_rmse_s_mean", "delta_rmse_s_ci_low", "delta_rmse_s_ci_high",
    "entry_order_tau_original_vs_sparse_mean",
    "active_set_jaccard_s_original_vs_sparse_mean",
    "sparse_component_count_s_mean", "sparse_df_diagnostic_s_mean"
  ),
  drop = FALSE
]
write.csv(primary_control_summary, output_path("primary_control_summary_table.csv"), row.names = FALSE)
cat("Aggregate diagnostics and primary-control summary saved.\n")


                     method group_weight_specification configurations
1    component_lasso_glmnet             not_applicable              8
2 parent_group_lasso_grpreg               equal_parent              8
3 parent_group_lasso_grpreg            sqrt_group_size              8
  configurations_f1_improved configurations_f1_decreased
1                          0                           8
2                          0                           7
3                          0                           7
  configurations_rmse_improved configurations_rmse_worsened median_delta_f1_s
1                            0                            8        -0.1875000
2                            1                            7        -0.1811047
3                            0                            8        -0.1250000
  median_delta_rmse_s median_entry_tau_original_vs_sparse
1           0.4936209                           0.3746663
2           0.2136932                           0.3386663
3     

## 11. Final baseline reproduction report

The same required check already ran before summaries. This section displays its saved result for every component-Lasso row in the current run. It verifies F1/RMSE at target `s`, parent counts at `s` and `2s`, and selected data/path fingerprints. It does not attempt to reproduce the original LAR/`lars` fits or all original inter-method agreement statistics.


In [ ]:
print(reproduction_report)
cat("All current component-Lasso rows passed the declared baseline checks.\n")


         stage matched_rows expected_current_rows archived_partial_path_rows
1 complete_run           40                    40                          0
  current_retried_rows all_checks_passed max_abs_f1_difference
1                    0              TRUE          2.775558e-16
  max_abs_rmse_difference max_abs_parent_count_difference_s
1            5.329071e-15                                 0
  max_abs_parent_count_difference_2s max_relative_noise_sd_difference
1                                  0                     1.964398e-15
  max_component_dimension_difference max_raw_density_difference
1                                  0                          0
  max_lambda_max_relative_difference max_requested_points_difference
1                       3.330669e-15                               0
  max_requested_ratio_difference
1                              0
All current component-Lasso rows passed the declared baseline checks.


## 12. Lightweight figures

These figures are diagnostic summaries of the penalty-sensitivity analysis. They compare configuration-level sparse-minus-original effects under component-wise Lasso and equal-parent group Lasso. They are intended to complement, not replace, the original experiment outputs.


In [ ]:
# =============================================================================
# 12. FIGURES (base R; no extra package dependency)
# =============================================================================

plot_comparison <- function(summary_df, metric, ylab, filename, reference = 0) {
  d1 <- summary_df[
    summary_df$path_specification == PRIMARY_PATH_SPECIFICATION &
      summary_df$method == "component_lasso_glmnet", , drop = FALSE
  ]
  d2 <- summary_df[
    summary_df$path_specification == PRIMARY_PATH_SPECIFICATION &
      summary_df$method == "parent_group_lasso_grpreg" &
      summary_df$group_weight_specification == "equal_parent", , drop = FALSE
  ]
  keys <- c("n", "p", "correlation", "scheme")
  merged <- merge(
    d1[, c(keys, paste0(metric, "_mean"))],
    d2[, c(keys, paste0(metric, "_mean"))],
    by = keys, suffixes = c("_component", "_group")
  )
  x <- merged[[paste0(metric, "_mean_component")]]
  y <- merged[[paste0(metric, "_mean_group")]]
  pdf(output_path(filename), width = 6.5, height = 6.0)
  plot(
    x, y, pch = 19,
    xlab = paste0("Component-wise Lasso: ", ylab),
    ylab = paste0("Parent-group Lasso: ", ylab),
    main = paste("Parent-group penalty sensitivity —", RUN_MODE)
  )
  abline(h = reference, v = reference, lty = 2)
  abline(a = 0, b = 1, lty = 3)
  dev.off()
}

plot_comparison(configuration_summary, "delta_f1_s", "sparse - original F1", "delta_f1_component_vs_group.pdf")
plot_comparison(configuration_summary, "delta_rmse_s", "sparse - original RMSE", "delta_rmse_component_vs_group.pdf")
plot_comparison(
  configuration_summary,
  "entry_order_tau_original_vs_sparse",
  "original-vs-sparse entry-order Kendall tau",
  "entry_order_preservation_component_vs_group.pdf",
  reference = 0
)
cat("Penalty-sensitivity diagnostic figures saved.\n")


png 
  2

png 
  2

png 
  2

Penalty-sensitivity diagnostic figures saved.


## 13. Interpretation and reporting

Use the full-run results to assess penalty sensitivity configuration by configuration.

- If a direction persists, report that it **persisted under the examined parent-group penalty**. Do not infer an independent effect of matrix sparsity or conclude that component-wise regularization had no role.
- If results change, identify the affected settings and qualify the corresponding component-Lasso interpretation.
- Report realized parent counts, component counts, target attainment, and the distinct definitions of the complexity diagnostics. Targets use the known support size, so this is an oracle-size simulation comparison rather than a data-driven tuning rule.
- Describe Kendall/Jaccard results as within-method representation preservation. This analysis does not reassess the original agreement among Lasso, LAR, and glmnet.
- Retain the condition that outcomes were generated by an original-feature linear model. The experiment does not settle performance under nonlinear or interval-specific signals.
- When all group sizes are equal, rescaling all group weights can be absorbed into the normalized lambda path. Identical weight-sensitivity results then provide no additional evidence.

For reproducible reporting, record the `grpreg` group-orthonormalization convention, group multipliers, normalized lambda grids, convergence retries, target-selection rule, and predictor-only DF conventions. Treat this notebook as a penalty-sensitivity extension of the empirical study; it does not define a new algorithm or theory.

The archived truncated glmnet path is audited separately. Inspect its rerun and any resulting aggregate changes before describing the earlier sensitivity paths as uniformly complete. This notebook does not overwrite the archived baseline outputs.


In [ ]:
# =============================================================================
# 13. SESSION INFORMATION AND OUTPUT MANIFEST
# =============================================================================

capture.output(sessionInfo(), file = output_path("session_info.txt"))

manifest <- data.frame(
  file = sort(list.files(OUTPUT_DIRECTORY, recursive = TRUE)),
  stringsAsFactors = FALSE
)
write.csv(manifest, output_path("generated_files_manifest.csv"), row.names = FALSE)
print(manifest)
writeLines(if (RUN_MODE == "quick")
  "QUICK VALIDATION PASSED. Restart R, set RUN_MODE to full, and run all cells for the complete analysis."
  else "FULL RUN COMPLETED with required coverage, path, and reproduction checks.",
  output_path("RUN_STATUS.txt"))
cat(if (RUN_MODE == "quick") "QUICK VALIDATION PASSED.\n" else "FULL RUN COMPLETED.\n")


                                                                 file
1                                           aggregate_diagnostics.csv
2                                     archived_glmnet_path_issues.csv
3                        archived_truncated_path_baseline_details.csv
4                         archived_truncated_path_baseline_report.csv
5                                   archived_truncated_path_rerun.csv
6      checkpoints/primary_controlled/scenario_01/replication_001.rds
7      checkpoints/primary_controlled/scenario_01/replication_002.rds
8      checkpoints/primary_controlled/scenario_11/replication_001.rds
9      checkpoints/primary_controlled/scenario_11/replication_002.rds
10 checkpoints/sensitivity_dense_wide/scenario_01/replication_001.rds
11 checkpoints/sensitivity_dense_wide/scenario_01/replication_002.rds
12 checkpoints/sensitivity_dense_wide/scenario_11/replication_001.rds
13 checkpoints/sensitivity_dense_wide/scenario_11/replication_002.rds
14                  

## Running and interpreting the notebook

1. Place this notebook in the project root next to `extended_monte_carlo_model_selection.ipynb`. Use the original R environment when possible: the archived experiment records R 4.4.1, MASS 7.3-60.2, and glmnet 4.1-10. Add `grpreg` if missing; avoid updating the baseline packages as part of that installation.
2. Restart the R kernel and execute all cells with the default `RUN_MODE <- "quick"`. The baseline results CSV must be available at `outputs/extended_monte_carlo/simulation_replication_results_controlled_paths.csv`.
3. Confirm the final `QUICK VALIDATION PASSED` message and inspect any retries, target overshoots/fallbacks, and singleton-calibration differences. If a check fails, retain the failure log and stop for diagnosis.
4. Set `RUN_MODE <- "full"`, restart R, and execute all cells. With both group-weight specifications enabled, this requests 48,000 main model paths plus preflight and retry fits. Outputs are written to separate `quick/<signature>/` and `full/<signature>/` directories under `outputs/parent_group_penalty_sensitivity/`.
5. Inspect complete configuration-level results, convergence diagnostics, target-attainment summaries, and baseline-reproduction checks before interpreting aggregate summaries.

For an existing Jupyter environment, run from the project root:

```bash
jupyter nbconvert --to notebook --execute \
  --ExecutePreprocessor.timeout=-1 \
  --output parent_group_penalty_sensitivity_quick_executed.ipynb \
  parent_group_penalty_sensitivity.ipynb
```

Use a different executed filename for the full run. Keep the complete configuration and attempt diagnostics, not just aggregate tables. No outcomes are assumed or prewritten in this notebook.
